# 🩺 Continuum — Clinical AI Platform

> **End-to-end clinical AI pipeline** — from patient intake to drug safety check, built on **MedGemma 4b**, LanceDB RAG, and a Gradio sequential UI.

---

## Overview

**Sequential pipeline flow:**
`Patient Registration` → `Clinical Interview` → `AI Triage` → `CXR Diagnosis` → `Drug Safety Check` → `Complete Records`

---

## Notebook Structure

| Cell | Module | Purpose |
|------|--------|---------|
| 2 | Imports | Library imports, HuggingFace login, device detection |
| 3 | Config | Project paths, constants, device/dtype, department-modality map |
| 4 | Utilities | File I/O, registry lookups, EHR helpers, record builders, FHIR R4 builders |
| 5 | Model Loaders | Lazy-load MedGemma 4b multimodal pipeline + medical embedding model |
| 6 | Vector Store | LanceDB CXR vector store — create/open + RAG search |
| 7 | Prompts | Structured prompt wrappers for intake, triage, CXR diagnosis, lab parsing |
| 8 | Algorithms | Core AI logic — interview, triage, CXR analysis, lab fusion, drug safety |
| 9 | LangGraph | 5-node agentic workflow graph with conditional CXR routing |
| 10 | CLI Runner | `run_pipeline()` — full CLI execution entry point |
| 11 | Gradio UI | `launch_gradio_ui()` — 6-tab sequential clinical UI |

---

## Quick Start

```python
# Run all cells 2 → 11 in order, then:
launch_gradio_ui()          # Opens UI at http://localhost:7860
```

---

## Key Design Points

- **Patient ID = mobile phone number** — unified key across all registries
- **Lazy model loading** — MedGemma loads on first use (or pre-loaded via `preload_pipeline()`)
- **CXR routing** — triage routes to imaging tab only when Pulmonology/Radiology is selected
- **Longitudinal mode** — patients with prior imaging visits get dual-image comparative analysis
- **AI Insight key** — imaging findings are concatenated into patient state and shown in all downstream steps
- **Drug safety two-layer validation architecture** — AI pharmacological review layer and fuzzy name matching + allergy cross-reactivity + DDI + pharmacogenomics (PGx) 
- **FHIR R4** — Patient / Condition / Observation / MedicationRequest bundle generated per visit
- **No external APIs** — fully offline; HF token only needed for model download

---

## Data Sources

| Source | Path | Description |
|--------|------|-------------|
| Patient Registry | `data/master/global_patient_registry.csv` | Demographics + gene profile |
| Provider Registry | `data/master/global_provider_registry.csv` | Doctors, slots, departments |
| Drug Knowledge | `Mod4/data/drug_knowledge.json` | DDI × 37, Allergy × 10, Contraind × 18, PGx × 6 |
| CXR Vectors | `Mod3/.lancedb/` | LanceDB CXR embeddings (ingested from JSONL on first run) |
| Session Records | `Mod1/data/conversations/` | Per-visit JSON + FHIR bundle + lab JSON |


## Cell 2 — Library Imports & Environment Setup

**Key points:**
- Imports all required standard and third-party libraries
- Auto-installs missing packages (`sentence-transformers`, `lancedb`, `langgraph`, `json-repair`, `scikit-image`)
- Loads `.env` file and authenticates with Hugging Face Hub using `HF_TOKEN`


In [ ]:

# ============================================================
# 2. Library Imports & Environment Configuration
# ============================================================

import os
import re
import sys
import json
import uuid
import random
import operator
import warnings
from enum import Enum
from dataclasses import dataclass, field
from pathlib import Path
from datetime import datetime
from typing import Annotated, TypedDict, List, Dict, Optional, Any

import numpy as np
import pandas as pd
from PIL import Image
import torch
from transformers import pipeline as hf_pipeline
from dotenv import load_dotenv
from huggingface_hub import login

# Sentence-transformers for CXR embeddings
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

# LanceDB — local CXR vector store
try:
    import lancedb
    import pyarrow as pa
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "lancedb", "pyarrow"])
    import lancedb
    import pyarrow as pa

# LangGraph — agentic workflow orchestration
try:
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "langgraph"])
    from langgraph.graph import StateGraph, END
    from langgraph.checkpoint.memory import MemorySaver

# json-repair — robust parsing of AI JSON outputs
try:
    import json_repair
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "json-repair"])
    import json_repair

# scikit-image — image padding for CXR preprocessing
try:
    from skimage import color as sk_color, util as sk_util
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-image"])
    from skimage import color as sk_color, util as sk_util

# ── Environment & HuggingFace login ──────────────────────────
load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")
if HF_TOKEN:
    login(token=HF_TOKEN)
else:
    warnings.warn(
        "HF_TOKEN not set — Hugging Face model downloads may be restricted. "
        "Add HF_TOKEN=<your_token> to a .env file in the project root.",
        stacklevel=1,
    )


## Cell 3 — Configuration & Project Paths

**Key points:**
- All directory paths derived from a single `ROOT` variable — easy to relocate the project
- Reads `Mod1/config.json` for sub-path overrides, with safe defaults
- `DEPARTMENT_TO_MODALITIES` controls which departments trigger CXR imaging in the pipeline
- `FUZZY_MATCH_THRESHOLD` (0.82) tunes drug-name matching sensitivity for safety checks


In [ ]:

# ============================================================
# 3. Constants, Project Paths & Runtime Handles
# ============================================================

import difflib  # stdlib — used for fuzzy drug name matching

from google.colab import drive
drive.mount('/content/drive')

# ── Directory roots ───────────────────────────────────────────
ROOT     = Path("/content/drive/MyDrive/ProjectC")
MOD1_DIR = ROOT / "CXRDiagnosis"
MOD3_DIR = ROOT / "PatientRegistration"
MOD4_DIR = ROOT / "Prescription_KG_Data"

# ── Intake module I/O paths (overridable via Mod1/config.json) ─
_mod1_cfg      = {}
_mod1_cfg_path = MOD1_DIR / "config.json"
if _mod1_cfg_path.exists():
    with open(_mod1_cfg_path) as _f:
        _mod1_cfg = json.load(_f)

MOD1_CONVERSATION_DIR = MOD1_DIR / _mod1_cfg.get("CONVERSATION_SAVE_DIR", "data/conversations")
MOD1_CSV_PATH         = MOD1_DIR / _mod1_cfg.get("CSV_SAVE_PATH",          "data/patient_records/records.csv")
MOD1_REPORTS_DIR      = MOD1_DIR / _mod1_cfg.get("reports_dir",            "data/reports")

# ── Global master registries ──────────────────────────────────
DATA_MASTER_DIR              = ROOT / "master"
GLOBAL_PATIENT_REGISTRY_CSV  = DATA_MASTER_DIR / "global_patient_registry.csv"
GLOBAL_PROVIDER_REGISTRY_CSV = DATA_MASTER_DIR / "global_provider_registry.csv"
PATIENT_MASTER_CSV           = DATA_MASTER_DIR / "patient_master.csv"

# ── Output directories ────────────────────────────────────────
FHIR_OUTPUT_DIR  = MOD1_CONVERSATION_DIR / "fhir"   # FHIR R4 bundle per visit
LAB_SESSIONS_DIR = MOD1_CONVERSATION_DIR / "labs"   # parsed lab value JSON per visit

# ── CXR vector store ──────────────────────────────────────────
LANCEDB_DIR    = MOD3_DIR / ".lancedb"
CXR_JSONL_PATH = MOD3_DIR / "context_dataset_10000.jsonl"

# ── Drug knowledge base ───────────────────────────────────────
DRUG_KNOWLEDGE_JSON = MOD4_DIR / "drug_knowledge.json"

# ── Model & inference configuration ──────────────────────────
EMBED_MODEL_PRIMARY  = "sentence-transformers/embeddinggemma-300m-medical"
EMBED_MODEL_FALLBACK = "BAAI/bge-small-en-v1.5"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32

# ── Drug-name fuzzy matching threshold (0–1 Sequence Matcher ratio) ──
FUZZY_MATCH_THRESHOLD = 0.82

# ── Department → imaging modality map (controls CXR routing) ─
DEPARTMENT_TO_MODALITIES = {
    "Pulmonology":      ["X-Ray"],
    "Radiology":        ["X-Ray"],
    "General Medicine": ["X-Ray"],
}

# ── Lazy runtime handles (populated by model/DB loaders) ─────
MEDGEMMA_MODEL = None
EMBED_MODEL    = None
LANCEDB_TABLE  = None
_DRUG_KB       = None


# -- Verification printout
for label, p in [
    ("GLOBAL_PATIENT_REGISTRY_CSV ",  GLOBAL_PATIENT_REGISTRY_CSV),
    ("GLOBAL_PROVIDER_REGISTRY_CSV",  GLOBAL_PROVIDER_REGISTRY_CSV),
    ("MOD1_CONVERSATION_DIR       ",  MOD1_CONVERSATION_DIR),
    ("MOD1_CSV_PATH               ",  MOD1_CSV_PATH),
    ("DRUG_KNOWLEDGE_JSON         ",  DRUG_KNOWLEDGE_JSON),
    ("CXR_JSONL_PATH              ",  CXR_JSONL_PATH),
    ("LANCEDB_DIR                 ",  LANCEDB_DIR),
    ("FHIR_OUTPUT_DIR             ",  FHIR_OUTPUT_DIR),
    ("LAB_SESSIONS_DIR            ",  LAB_SESSIONS_DIR),
]:
    exists = "exists" if Path(p).exists() else "will be created"
    print(f"  {label}  [{exists}]  {p}")
print(f"  Device: {DEVICE}  |  dtype: {DTYPE}")
print(f"  Fuzzy match threshold: {FUZZY_MATCH_THRESHOLD}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  GLOBAL_PATIENT_REGISTRY_CSV   [exists]  /content/drive/MyDrive/ProjectC/master/global_patient_registry.csv
  GLOBAL_PROVIDER_REGISTRY_CSV  [exists]  /content/drive/MyDrive/ProjectC/master/global_provider_registry.csv
  MOD1_CONVERSATION_DIR         [exists]  /content/drive/MyDrive/ProjectC/CXRDiagnosis/data/conversations
  MOD1_CSV_PATH                 [exists]  /content/drive/MyDrive/ProjectC/CXRDiagnosis/data/patient_records/records.csv
  DRUG_KNOWLEDGE_JSON           [exists]  /content/drive/MyDrive/ProjectC/Prescription_KG_Data/drug_knowledge.json
  CXR_JSONL_PATH                [exists]  /content/drive/MyDrive/ProjectC/PatientRegistration/context_dataset_10000.jsonl
  LANCEDB_DIR                   [will be created]  /content/drive/MyDrive/ProjectC/PatientRegistration/.lancedb
  FHIR_OUTPUT_DIR               [will be created]  /content/drive/MyDrive/Pro

## Cell 4 — Core Utility Functions

**Key points:**
- **4-A File I/O** — `read_json`, `save_json`, `read_txt`, `save_txt`, `make_directory`
- **4-B Registry lookups** — patient + provider registries; free-slot finder; slot booking
- **4-C EHR helpers** — `get_patient_details`, `get_ehr_summary` (via MedGemma)
- **4-D Record builders** — `build_patient_json_record` (canonical visit record), CSV row builder
- **4-E JSON parsing** — `safe_parse_json` with `json-repair` fallback for AI outputs
- **4-F FHIR R4 builders** — Patient / Condition / Observation / MedicationRequest / Bundle


In [3]:

# ============================================================
# 4. Core Utility Functions
# ============================================================
# 4-A  File & I/O helpers
# 4-B  Global registry lookups  (patient + provider)
# 4-C  EHR / patient record helpers
# 4-D  Canonical record builders
# 4-E  JSON parsing helpers
# 4-F  FHIR R4 resource builders
# ============================================================

# ── 4-A  File & I/O ──────────────────────────────────────────

def make_directory(path) -> None:
    os.makedirs(str(path), exist_ok=True)

def read_json(path) -> dict:
    try:
        with open(path, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        warnings.warn(f"[WARN] File not found: {path}", stacklevel=2)
        return {}
    except json.JSONDecodeError:
        warnings.warn(f"[WARN] Invalid JSON in: {path}", stacklevel=2)
        return {}

def save_json(data: dict, path) -> None:
    make_directory(Path(path).parent)
    with open(path, "w") as f:
        json.dump(data, f, indent=2)

def read_txt(path) -> str:
    with open(path, "r") as f:
        return f.read()

def save_txt(data: str, path) -> None:
    make_directory(Path(path).parent)
    with open(path, "w") as f:
        f.write(data)

def ask_model(model, dialog: list) -> str:
    """Single inference call wrapper for the MedGemma hf_pipeline."""
    output = model(dialog)
    return output[0]["generated_text"][-1]["content"]


# ── 4-B  Global registry lookups ─────────────────────────────

def _load_registry(csv_path: Path) -> pd.DataFrame:
    if not csv_path.exists():
        warnings.warn(f"[WARN] Registry not found: {csv_path}", stacklevel=2)
        return pd.DataFrame()
    return pd.read_csv(csv_path, dtype=str)

def load_patient_registry()  -> pd.DataFrame:
    return _load_registry(GLOBAL_PATIENT_REGISTRY_CSV)

def load_provider_registry() -> pd.DataFrame:
    return _load_registry(GLOBAL_PROVIDER_REGISTRY_CSV)

def lookup_patient_by_phone(phone: str) -> Optional[Dict]:
    """Return matching patient registry row dict, or None if not found."""
    df = load_patient_registry()
    if df.empty or "phone" not in df.columns:
        return None
    rows = df[df["phone"].str.strip() == phone.strip()]
    return rows.iloc[0].to_dict() if not rows.empty else None

def lookup_patient_by_id(pid: str) -> Optional[Dict]:
    df = load_patient_registry()
    if df.empty or "global_patient_id" not in df.columns:
        return None
    rows = df[df["global_patient_id"].str.strip() == pid.strip()]
    return rows.iloc[0].to_dict() if not rows.empty else None

def lookup_providers_by_department(dept: str) -> pd.DataFrame:
    df = load_provider_registry()
    if df.empty or "department_name" not in df.columns:
        return pd.DataFrame()
    mask = df["department_name"].str.strip().str.lower() == dept.strip().lower()
    return df[mask].copy()

def get_free_slots_from_registry(dept: str) -> List[Dict]:
    """Return unbooked provider slots for a department."""
    df = lookup_providers_by_department(dept)
    if df.empty or "is_booked" not in df.columns:
        return []
    free = df[df["is_booked"].astype(str).str.strip().isin(["0", "False", "false", ""])].copy()
    slots = []
    for _, r in free.iterrows():
        slots.append({
            "slot_id":       r.get("slot_id", ""),
            "doctor_id":     r.get("doctor_id", ""),
            "doctor_name":   r.get("doctor_name", ""),
            "hospital_id":   r.get("hospital_id", ""),
            "hospital_name": r.get("hospital_name", ""),
            "department":    r.get("department_name", dept),
            "time":          f"{r.get('available_date','')} {r.get('start_time','')}".strip(),
            "modalities":    r.get("modalities", ""),
        })
    return slots

def get_any_free_slot() -> Optional[Dict]:
    """Return any one unbooked slot across all providers."""
    df = load_provider_registry()
    if df.empty or "is_booked" not in df.columns:
        return None
    free = df[df["is_booked"].astype(str).str.strip().isin(["0", "False", "false", ""])].copy()
    if free.empty:
        return None
    r = free.iloc[0]
    return {
        "slot_id":       r.get("slot_id", ""),
        "doctor_id":     r.get("doctor_id", ""),
        "doctor_name":   r.get("doctor_name", ""),
        "hospital_name": r.get("hospital_name", ""),
        "department":    r.get("department_name", ""),
        "time":          f"{r.get('available_date','')} {r.get('start_time','')}".strip(),
    }

def mark_slot_booked(slot_id) -> bool:
    """Mark a slot as booked (is_booked=1) in the provider registry CSV."""
    if not GLOBAL_PROVIDER_REGISTRY_CSV.exists():
        return False
    df = pd.read_csv(GLOBAL_PROVIDER_REGISTRY_CSV, dtype=str)
    if "slot_id" not in df.columns:
        return False
    mask = df["slot_id"].astype(str).str.strip() == str(slot_id).strip()
    if not mask.any():
        return False
    df.loc[mask, "is_booked"] = "1"
    df.to_csv(GLOBAL_PROVIDER_REGISTRY_CSV, index=False)
    return True

def get_all_departments() -> List[str]:
    df = load_provider_registry()
    if df.empty or "department_name" not in df.columns:
        return []
    return df["department_name"].dropna().str.strip().unique().tolist()

def confirm_booking(patient_id: str, slot: Dict) -> str:
    """Mark slot booked and write appointment record. Returns appointment ID."""
    appt_id = f"APT-{str(uuid.uuid4())[:8]}"
    mark_slot_booked(slot.get("slot_id"))
    appt_row = {
        "appt_id":    appt_id,
        "patient_id": patient_id,
        "doctor_id":  slot.get("doctor_id"),
        "slot_id":    slot.get("slot_id"),
        "status":     "Scheduled",
        "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }
    appt_csv = MOD1_DIR / "data" / "appointments.csv"
    make_directory(appt_csv.parent)
    if appt_csv.exists():
        existing = pd.read_csv(appt_csv)
        pd.concat([existing, pd.DataFrame([appt_row])], ignore_index=True).to_csv(appt_csv, index=False)
    else:
        pd.DataFrame([appt_row]).to_csv(appt_csv, index=False)
    return appt_id


# ── 4-C  EHR / patient record helpers ────────────────────────

def collect_new_patient_details(patient_id: str) -> dict:
    """CLI helper — prompts for new patient name/age/sex. Not used in Gradio UI."""
    return {
        "patient_id": patient_id,
        "name":       input("  Your full name: ").strip(),
        "age":        input("  Your age: ").strip(),
        "sex":        input("  Your sex (M/F/Other): ").strip(),
    }

def get_patient_details(patient_phone: str) -> tuple:
    """
    Resolve patient details + EHR visit history from registries.
    Returns: (patient_details dict, last_visit int, ehr_records, encounter_df)
    """
    patient_details = {}
    patient_id      = patient_phone

    reg_row = lookup_patient_by_phone(patient_phone)
    if reg_row:
        patient_id       = reg_row.get("global_patient_id", patient_phone)
        gene_profile_raw = reg_row.get("gene_profile", "{}")
        try:
            gene_profile = json.loads(gene_profile_raw) if isinstance(gene_profile_raw, str) else {}
        except Exception:
            gene_profile = {}

        patient_details = {
            "patient_id":          patient_id,
            "name":                reg_row.get("patient_name", ""),
            "age":                 reg_row.get("age", ""),
            "sex":                 reg_row.get("sex", ""),
            "phone":               reg_row.get("phone", patient_phone),
            "primary_dept":        reg_row.get("primary_department", ""),
            "doctor_name":         reg_row.get("primary_doctor_name", ""),
            "hospital":            reg_row.get("primary_hospital_name", ""),
            "allergies":           [],
            "current_medications": [],
            "conditions":          [],
            "renal_function":      {"egfr": 999, "status": "NORMAL"},
            "gene_profile":        gene_profile,
        }
    else:
        patient_details = collect_new_patient_details(patient_id)
        patient_details.update({
            "allergies": [], "current_medications": [],
            "conditions": [], "renal_function": {"egfr": 999, "status": "NORMAL"},
            "gene_profile": {},
        })

    last_visit  = 0
    ehr_records = "There is No Past Records"
    df          = pd.DataFrame()

    if MOD1_CSV_PATH.exists():
        df = pd.read_csv(MOD1_CSV_PATH)
        if "patient_id" in df.columns:
            df["patient_id"] = df["patient_id"].astype(str)
            rows = df[df["patient_id"] == str(patient_id)]
        else:
            rows = pd.DataFrame()

        if not rows.empty and {"visit", "report_path"}.issubset(rows.columns):
            rows["visit"] = pd.to_numeric(rows["visit"], errors="coerce")
            rows = rows.dropna(subset=["visit"])
            if not rows.empty:
                ehr_records = [
                    (int(v), read_txt(rp))
                    for v, rp in zip(rows["visit"].tolist(), rows["report_path"].tolist())
                    if isinstance(rp, str) and os.path.exists(rp)
                ] or "There is No Past Records"
                last_visit = int(rows["visit"].max())

    return patient_details, last_visit, ehr_records, df


def detect_cxr_mode(patient_id: str) -> str:
    """
    Return 'longitudinal' if the patient has a prior visit with imaging recorded,
    otherwise return 'single'. Checks multiple CSV sources with safe fallbacks.
    """
    for csv_path in (PATIENT_MASTER_CSV, MOD1_CSV_PATH):
        if not csv_path.exists():
            continue
        try:
            df = pd.read_csv(csv_path, dtype=str)
            if "patient_id" not in df.columns:
                continue
            rows = df[df["patient_id"].str.strip() == str(patient_id).strip()]
            if rows.empty:
                continue
            for img_col in ("recommended_imaging", "imaging_paths", "cxr_paths"):
                if img_col in rows.columns:
                    if (rows[img_col].dropna().str.strip().str.len() > 2).any():
                        return "longitudinal"
            if "selected_departments" in rows.columns:
                cxr_depts = set(DEPARTMENT_TO_MODALITIES.keys())
                for dept_str in rows["selected_departments"].dropna():
                    for dept in cxr_depts:
                        if dept.lower() in dept_str.lower():
                            return "longitudinal"
        except Exception:
            continue
    return "single"


def get_ehr_summary(patient_name: str, ehr_records, model) -> str:
    """Summarise EHR visit records with MedGemma, or return no-records string."""
    if ehr_records == "There is No Past Records":
        return ehr_records
    dialog = [
        {"role": "system", "content": [{"type": "text", "text": (
            f"You are a medical assistant summarising EHR records for {patient_name}. "
            "Provide a concise clinical summary: conditions, medications, past treatments. "
            "Facts only. Data format: list of (visit_id, report_text) tuples."
        )}]},
        {"role": "user", "content": [{"type": "text", "text": str(ehr_records)}]},
    ]
    return ask_model(model, dialog)


# ── 4-D  Canonical record builders ───────────────────────────

def build_patient_json_record(
    *,
    patient_id      : str,
    patient_details : dict,
    ehr_summary     : str  = "",
    interview       : dict = None,
    report_summary  : str  = "",
    triage          : dict = None,
    booking         : dict = None,
    imaging         : dict = None,
    drug_safety     : dict = None,
    visit           : int  = 0,
    date_time       : str  = "",
) -> dict:
    """Per-visit JSON record — single source of truth passed through all pipeline steps."""
    return {
        "meta": {
            "session_id": str(uuid.uuid4()),
            "patient_id": patient_id,
            "visit":      visit,
            "date_time":  date_time or datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        },
        "demographics":   patient_details,
        "ehr_summary":    ehr_summary,
        "interview":      interview    or {},
        "report_summary": report_summary,
        "triage":         triage       or {},
        "booking":        booking      or {},
        "imaging":        imaging      or {},
        "drug_safety":    drug_safety  or {},
    }

def build_patient_row_record(
    *, patient_id: str, date_time: str, visit: int,
    conversation_path: str, report_path: str,
    status: str = "intake_complete", priority: str = "normal",
    target_department: Optional[str] = None,
) -> dict:
    return {
        "patient_id":        patient_id, "date_time":  date_time,
        "visit":             visit,      "status":     status,
        "priority":          priority,   "target_department": target_department,
        "conversation_path": conversation_path,
        "report_path":       report_path,
    }


# ── 4-E  JSON parsing helpers ─────────────────────────────────

def extract_first_json(text: str) -> Optional[str]:
    """Extract the first {...} JSON block from a string."""
    try:
        s, e = text.find("{"), text.rfind("}")
        return text[s: e + 1] if s != -1 and e != -1 else None
    except Exception:
        return None

def safe_parse_json(text: str) -> Optional[dict]:
    """Try json.loads then json_repair.loads, returning None on total failure."""
    snippet = extract_first_json(text)
    if not snippet:
        return None
    try:
        return json.loads(snippet)
    except json.JSONDecodeError:
        try:
            return json_repair.loads(snippet)
        except Exception:
            return None


# ── 4-F  FHIR R4 resource builders ───────────────────────────

def build_fhir_patient_resource(patient_details: dict) -> dict:
    """FHIR R4 Patient resource from patient_details dict."""
    name   = patient_details.get("name", "Unknown")
    parts  = name.split(None, 1)
    given  = [parts[0]] if parts else []
    family = parts[1] if len(parts) > 1 else ""
    return {
        "resourceType": "Patient",
        "id": str(patient_details.get("patient_id", "")),
        "name": [{"use": "official", "family": family, "given": given}],
        "telecom": [{"system": "phone", "value": str(patient_details.get("phone", ""))}],
        "gender": {"M": "male", "F": "female"}.get(
            str(patient_details.get("sex", "")).upper(), "unknown"),
        "birthDate": patient_details.get("date_of_birth", ""),
    }

def build_fhir_condition_resource(ddx_entry: dict, patient_id: str) -> dict:
    """FHIR R4 Condition resource from a DDx entry (keys: condition, icd10, probability, evidence)."""
    return {
        "resourceType": "Condition",
        "subject": {"reference": f"Patient/{patient_id}"},
        "code": {"coding": [{
            "system":  "http://hl7.org/fhir/sid/icd-10",
            "code":    ddx_entry.get("icd10", ""),
            "display": ddx_entry.get("condition", ""),
        }]},
        "note": [{"text": f"Probability: {ddx_entry.get('probability','')} — {ddx_entry.get('evidence','')}"}],
        "verificationStatus": {"coding": [{"system":
            "http://terminology.hl7.org/CodeSystem/condition-ver-status", "code": "provisional"}]},
    }

def build_fhir_observation_resource(lab_entry: dict, patient_id: str) -> dict:
    """FHIR R4 Observation resource from a parsed lab entry."""
    return {
        "resourceType": "Observation",
        "status": "final",
        "subject": {"reference": f"Patient/{patient_id}"},
        "code": {"coding": [{"system": "http://loinc.org", "display": lab_entry.get("name", "")}]},
        "valueQuantity": {
            "value": lab_entry.get("value", ""),
            "unit":  lab_entry.get("unit", ""),
        },
        "referenceRange": [{"text": lab_entry.get("reference_range", "")}],
        "interpretation": [{"coding": [{"display": lab_entry.get("status", "normal")}]}],
    }

def build_fhir_medication_request_resource(drug: str, dosage: str,
                                           safety_result: dict, patient_id: str) -> dict:
    """FHIR R4 MedicationRequest from drug safety output."""
    status = "active" if safety_result.get("status") == "PASS" else "stopped"
    return {
        "resourceType": "MedicationRequest",
        "status": status,
        "intent": "proposal",
        "subject": {"reference": f"Patient/{patient_id}"},
        "medicationCodeableConcept": {"coding": [{"display": drug}]},
        "dosageInstruction": [{"text": dosage}],
        "note": [{"text": (
            f"Safety status: {safety_result.get('status','--')} | "
            f"Alert: {safety_result.get('alert_level','--')}"
        )}],
    }

def build_fhir_bundle(patient_id: str, session_id: str, *resources) -> str:
    """
    Assemble a FHIR R4 Bundle of type 'collection' and save to FHIR_OUTPUT_DIR.
    Returns the saved file path string.
    """
    bundle = {
        "resourceType": "Bundle",
        "id": f"{patient_id}-{session_id}",
        "type": "collection",
        "timestamp": datetime.now().strftime("%Y-%m-%dT%H:%M:%SZ"),
        "entry": [{"resource": r} for r in resources if r],
    }
    make_directory(FHIR_OUTPUT_DIR)
    out_path = FHIR_OUTPUT_DIR / f"{patient_id}_{session_id}_bundle.json"
    save_json(bundle, str(out_path))
    return str(out_path)


## Cell 5 — Model Loaders

**Key points:**
- `load_medgemma()` — loads the MedGemma 4b multimodal `image-text-to-text` pipeline
- `load_embed_model()` — loads the medical sentence-embedding model (with fallback to BAAI/bge)
- Both loaders are **no-ops if already loaded** (safe to call multiple times)
- `get_medgemma()` / `get_embed_model()` — lazy accessors used throughout the pipeline


In [4]:

# ============================================================
# 5. Model Loaders — MedGemma 4b + CXR Embedding Model (lazy)
# ============================================================

def load_medgemma() -> None:
    """Load MedGemma 4b multimodal image-text-to-text pipeline. No-op if already loaded."""
    global MEDGEMMA_MODEL
    if MEDGEMMA_MODEL is not None:
        return
    print(f"Loading MedGemma 4b on {DEVICE} …")
    MEDGEMMA_MODEL = hf_pipeline(
        "image-text-to-text",
        model="google/medgemma-4b-it",
        torch_dtype=DTYPE,
        device=DEVICE,
        token=HF_TOKEN,
    )
    MEDGEMMA_MODEL.model.generation_config.do_sample = False
    print("✓ MedGemma 4b loaded")


def load_embed_model() -> None:
    """Load medical sentence-embedding model for CXR vector search. No-op if already loaded."""
    global EMBED_MODEL
    if EMBED_MODEL is not None:
        return
    print(f"Loading embedding model on {DEVICE} …")
    try:
        EMBED_MODEL = SentenceTransformer(EMBED_MODEL_PRIMARY, device=DEVICE)
        print(f"✓ Embed model: {EMBED_MODEL_PRIMARY}")
    except Exception as e:
        print(f"  ⚠  Primary model failed ({e}), loading fallback …")
        EMBED_MODEL = SentenceTransformer(EMBED_MODEL_FALLBACK, device=DEVICE)
        print(f"✓ Embed model (fallback): {EMBED_MODEL_FALLBACK}")


def get_medgemma():
    """Return MEDGEMMA_MODEL, loading on first call."""
    if MEDGEMMA_MODEL is None:
        load_medgemma()
    return MEDGEMMA_MODEL


def get_embed_model():
    """Return EMBED_MODEL, loading on first call."""
    if EMBED_MODEL is None:
        load_embed_model()
    return EMBED_MODEL


## Cell 6 — CXR Vector Store (LanceDB)

**Key points:**
- `open_or_create_lancedb()` — opens an existing LanceDB table or creates it from JSONL on first run
- Filters JSONL to **CXR/chest X-ray records only** during ingestion
- `search_cxr_context(query, top_k)` — cosine similarity search, returns top-k similar reports
- Gracefully handles empty/uninitialised table — returns `[]` without raising exceptions
- Must be called before `run_pipeline()` (done automatically in Cell 10)


In [5]:

# ============================================================
# 6. LanceDB CXR Vector Store
# ============================================================
# Stores CXR report-text embeddings for RAG-based diagnosis.
# On first run  -> creates table + ingests context_dataset_10000.jsonl (CXR only).
# On subsequent -> opens the existing table.
# ============================================================

def _get_embed_dim() -> int:
    return get_embed_model().get_sentence_embedding_dimension()

def embed_texts(texts: List[str], batch_size: int = 64) -> np.ndarray:
    return get_embed_model().encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )

def _lancedb_schema(dim: int) -> pa.Schema:
    return pa.schema([
        pa.field("id",             pa.string()),
        pa.field("vector",         pa.list_(pa.float32(), dim)),
        pa.field("source_dataset", pa.string()),
        pa.field("study_date",     pa.string()),
        pa.field("protocol",       pa.string()),
        pa.field("modality",       pa.string()),
        pa.field("text",           pa.string()),
    ])

def _iter_jsonl(path: Path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    yield json.loads(line)
                except Exception:
                    continue

def _prepare_cxr_batch(records: list) -> tuple:
    """Extract CXR-only records from a raw JSONL batch."""
    ids, texts, metas = [], [], []
    for rec in records:
        meta     = rec.get("record_metadata", {})
        rid      = meta.get("dataset_row_id") or meta.get("source_id")
        text     = rec.get("vector_embedding_text") or ""
        modality = meta.get("modality", "").upper().replace("-", "").replace(" ", "")
        # Keep only CXR / CHESTXRAY / XRAY records
        if not rid or not text:
            continue
        if modality not in ("CXR", "CHESTXRAY", "XRAY", "X_RAY"):
            continue
        ids.append(str(rid))
        texts.append(text)
        metas.append({
            "source_dataset": meta.get("source_dataset", ""),
            "study_date":     meta.get("study_date", ""),
            "protocol":       meta.get("protocol", ""),
            "modality":       "CXR",
            "text":           text,
        })
    return ids, texts, metas

def _ingest_cxr_jsonl(table, jsonl_path: Path, batch_size: int = 500) -> int:
    if not jsonl_path.exists():
        warnings.warn(f"JSONL not found: {jsonl_path}")
        return 0
    buffer, total = [], 0
    for rec in _iter_jsonl(jsonl_path):
        buffer.append(rec)
        if len(buffer) >= batch_size:
            ids, texts, metas = _prepare_cxr_batch(buffer)
            if ids:
                vecs = embed_texts(texts)
                table.add([{"id": ids[i], "vector": vecs[i].tolist(), **metas[i]} for i in range(len(ids))])
                total += len(ids)
                print(f"  Ingested {total} CXR records ...", end="\r")
            buffer = []
    if buffer:
        ids, texts, metas = _prepare_cxr_batch(buffer)
        if ids:
            vecs = embed_texts(texts)
            table.add([{"id": ids[i], "vector": vecs[i].tolist(), **metas[i]} for i in range(len(ids))])
            total += len(ids)
    print(f"\n  CXR records ingested: {total}")
    return total

def open_or_create_lancedb():
    """Open existing LanceDB table or create and ingest from JSONL. Sets LANCEDB_TABLE."""
    global LANCEDB_TABLE
    load_embed_model()
    dim    = _get_embed_dim()
    schema = _lancedb_schema(dim)
    db     = lancedb.connect(str(LANCEDB_DIR))

    if "context_vectors" in db.table_names():
        tbl   = db.open_table("context_vectors")
        count = len(tbl)
        print(f"Opened LanceDB table 'context_vectors' ({count} existing rows)")
    else:
        print("  Creating LanceDB table 'context_vectors' ...")
        tbl   = db.create_table("context_vectors", schema=schema, mode="overwrite")
        count = _ingest_cxr_jsonl(tbl, CXR_JSONL_PATH)
        print(f"LanceDB table created -- {count} CXR records indexed")

    LANCEDB_TABLE = tbl
    return tbl

def search_cxr_context(query_text: str, top_k: int = 3) -> List[Dict]:
    """
    Return top-k similar CXR context records for RAG prompt assembly.
    Returns empty list (never raises) if LanceDB is uninitialised, empty, or errors mid-search.
    """
    if LANCEDB_TABLE is None:
        warnings.warn("LanceDB not initialised — RAG context will be empty.")
        return []
    try:
        table_count = len(LANCEDB_TABLE)
        if table_count == 0:
            warnings.warn("LanceDB table is empty — RAG context will be empty.")
            return []
        qvec    = embed_texts([query_text], batch_size=1)[0].tolist()
        results = (
            LANCEDB_TABLE.search(qvec, vector_column_name="vector")
            .metric("cosine")
            .where("modality = 'CXR'")
            .limit(top_k)
            .to_pandas()
        )
        if results.empty:
            return []
        return [
            {"id": r.get("id",""), "source_dataset": r.get("source_dataset",""),
             "study_date": r.get("study_date",""), "protocol": r.get("protocol",""),
             "text": r.get("text","")}
            for _, r in results.iterrows()
        ]
    except Exception as exc:
        warnings.warn(f"LanceDB search failed ({exc}) — continuing without RAG context.")
        return []


## Cell 7 — Prompt Wrappers

**Key points:**
- Each pipeline stage has a dedicated system + user prompt builder, keeping logic cleanly separated
- **Intake prompt** — enforces one question per message, anatomical location follow-up rule, max 20 Qs
- **CXR diagnosis prompt** — injects RAG context from similar prior cases; requests structured JSON output
- **Longitudinal prompt** — compares multiple images chronologically; adds `change` field (new / improved / stable / worsened)
- **Lab DDx fusion prompt** — combines CXR differential, lab results, and EHR to produce unified ranked diagnosis
- All AI outputs are expected as **strict JSON** — parsed by `safe_parse_json` with repair fallback


In [6]:

# ============================================================
# 7. Prompt Wrappers  (one function per module feature)
# ============================================================

# ── Mod1 – Patient Intake ────────────────────────────────────

def prompt_intake_system(patient_name: str, ehr_summary: str) -> str:
    return f"""SYSTEM DIRECTIVE: Reason internally. Do not reveal reasoning. Follow all rules exactly.

### ROLE ###
You are a clinical intake assistant interviewing {patient_name} to collect structured information for their doctor. Your task is strictly information gathering.

### ABSOLUTE RULES ###
- Do NOT provide medical advice, diagnosis, reassurance, or clinical judgment.
- Ask only ONE question per message, 20 words or fewer.
- Do NOT number or label questions.
- Standard interview: up to 20 questions.
- Emergency (trauma / chest pain / breathing difficulty / loss of consciousness): max 5 questions, focused on event, timing, severity, current symptoms, immediate risks. End after question 5.

### ANATOMICAL LOCATION RULE ###
When patient reports pain in any body region, you MUST ask a follow-up that lists at least FOUR specific anatomical location options in the same question. Example for knee: "Is the pain front, inner side, outer side, or back of the knee?"

### INTERVIEW FLOW ###
1. Begin with exactly: "Thank you for booking an appointment. I am an assistant here to help your doctor prepare. To start, what is your main concern today?"
2. Ask one question at a time, using EHR context to avoid repeating known information.
3. End with exactly: "Thank you for answering my questions. I have everything needed to prepare a report for your visit. End interview."

### PATIENT EHR ###
{ehr_summary}
"""


# ── Mod1 – Report Writer ─────────────────────────────────────

def prompt_report_writer_system(ehr_summary: str) -> str:
    return f"""<role>Senior medical documentation assistant.</role>
<task>Generate a concise clinical intake report for the Primary Care Physician based on a patient interview and EHR.</task>
<principles>
1. Brevity: use medical terminology; omit pleasantries and filler.
2. Clinical relevance: prioritise History of Present Illness (onset, duration, quality, severity, timing, modifying factors). Include pertinent negatives. Filter EHR to only relevant history.
</principles>
<constraints>
- Facts only. No diagnosis or clinical assessment.
- Output ONLY the Markdown report text -- no preamble, no explanation.
</constraints>
<ehr_data>{ehr_summary}</ehr_data>"""

def prompt_report_writer_user(interview_dict: dict, ehr_summary: str) -> str:
    return f"""<interview>
{json.dumps(interview_dict, indent=2)}
</interview>
<previous_report>{ehr_summary}</previous_report>
<task>Update the report using new interview information. Integrate new symptoms, replace outdated details, maintain conciseness, preserve critical history, keep existing section titles.</task>
Generate the complete updated Markdown report now."""


# ── Mod2 – AI Triage ─────────────────────────────────────────

def prompt_triage_system(departments: List[str], imaging_tests: List[str], lab_tests: List[str]) -> str:
    return f"""You are a clinical triage specialist.

VALID DEPARTMENTS : {', '.join(departments)}
VALID IMAGING TESTS: {', '.join(imaging_tests)}
VALID LAB TESTS   : {', '.join(lab_tests)}

Return STRICT JSON only -- no prose, no markdown fences:
{{
  "triage_priority": "EMERGENT | URGENT | STABLE",
  "departments": ["<department_name>"],
  "recommended_imaging": [{{"name": "<test>", "priority": "STAT | ROUTINE", "reason": "<brief>", "confidence": 0.0}}],
  "recommended_labs":    [{{"name": "<test>", "priority": "STAT | ROUTINE", "reason": "<brief>", "confidence": 0.0}}],
  "clinical_summary": "<one sentence clinical rationale>"
}}

Rules:
- departments must be from VALID DEPARTMENTS only.
- If no imaging/labs are warranted, return empty arrays.
- clinical_summary must be under 30 words."""

def prompt_triage_user(patient_name: str, clinical_history: str, report_summary: str) -> str:
    return f"""Patient: {patient_name}
Clinical history: {clinical_history}
Intake report: {report_summary}

Provide your triage JSON now."""


# ── Mod3 – CXR Diagnosis (RAG) — structured JSON output ──────

def _cxr_guidance() -> List[str]:
    return [
        "Check positioning, rotation, and exposure quality.",
        "Assess lung fields (opacities, consolidation, hyperinflation).",
        "Evaluate cardiomediastinal silhouette width and borders.",
        "Check pleural spaces for effusion or pneumothorax.",
        "Inspect bony structures and soft tissues.",
        "Distinguish device/artifact from true pathology.",
        "Prioritise acute findings: airspace opacity, effusions, pneumothorax, pulmonary oedema.",
    ]

def prompt_cxr_diagnosis(retrieved_cases: List[Dict], anatomy: str = "Thorax", instruction: str = "") -> str:
    cases_block = ""
    for i, c in enumerate(retrieved_cases[:3], 1):
        cases_block += (
            f"\nCase #{i} | protocol: {c.get('protocol','')} | date: {c.get('study_date','')}\n"
            f"Key findings: {c.get('text','')}\n"
        )
    guidance = "\n".join(f"- {g}" for g in _cxr_guidance())
    task     = instruction or "Identify the primary diagnosis. Describe key findings, regions of interest, and differential diagnoses."
    return f"""Role  : Senior Radiologist
Task  : Analyse the provided Chest X-Ray (CXR) image. Focus on: {anatomy}.

Similar prior CXR cases (for context):
{cases_block if cases_block else '(no prior cases retrieved)'}

Radiological guidance:
{guidance}

Specific instruction: {task}

Output: Return STRICT JSON only -- no prose, no markdown fences:
{{
  "findings": "<narrative description of all visible findings>",
  "regions_of_interest": [
    {{"label": "<pathology name>", "location": "<anatomical location>", "size": "<approximate size or extent>"}}
  ],
  "ddx": [
    {{"condition": "<diagnosis name>", "icd10": "<ICD-10 code>", "probability": <0.0-1.0>, "evidence": "<supporting image features>"}}
  ]
}}

Rules:
- ddx must be sorted by probability descending.
- regions_of_interest is empty array [] if no focal abnormality detected.
- Every ddx entry must reference specific visible image features in the evidence field.
- Provide at least 1 and at most 5 ddx entries."""

def prompt_longitudinal_cxr(num_images: int) -> str:
    return f"""Compare the provided {num_images} Chest X-Ray images from the same patient in chronological order.

Output: Return STRICT JSON only -- no prose, no markdown fences:
{{
  "findings": "<overall narrative comparing all images>",
  "regions_of_interest": [
    {{"label": "<finding>", "location": "<anatomical location>", "size": "<size or extent>", "change": "new | improved | stable | worsened"}}
  ],
  "ddx": [
    {{"condition": "<diagnosis>", "icd10": "<ICD-10>", "probability": <0.0-1.0>, "evidence": "<progression evidence from images>"}}
  ],
  "progression_summary": "<one sentence describing overall trajectory: improving / worsening / stable>"
}}"""


# ── Mod3 – Lab Result Parser ──────────────────────────────────

def prompt_lab_parser(raw_lab_text: str) -> str:
    return f"""You are a clinical data extraction assistant.

Extract all lab test results from the text below.

Return STRICT JSON only -- no prose, no markdown fences:
{{
  "labs": [
    {{
      "name": "<test name>",
      "value": "<numeric or categorical value>",
      "unit": "<unit of measure>",
      "reference_range": "<normal range if stated>",
      "status": "normal | high | low | critical"
    }}
  ]
}}

If no lab results are found, return: {{"labs": []}}

Lab text:
{raw_lab_text}"""

def prompt_lab_ddx_fusion(lab_dict: dict, cxr_ddx: list, ehr_summary: str) -> str:
    lab_str = json.dumps(lab_dict.get("labs", []), indent=2) if lab_dict else "[]"
    cxr_str = json.dumps(cxr_ddx, indent=2) if cxr_ddx else "[]"
    return f"""You are a senior clinician performing diagnostic fusion.

Integrate the following data sources to produce a unified ranked differential diagnosis.

EHR SUMMARY:
{ehr_summary or "(none)"}

CXR DIFFERENTIAL DIAGNOSES:
{cxr_str}

LAB RESULTS:
{lab_str}

Return STRICT JSON only -- no prose, no markdown fences:
{{
  "fused_ddx": [
    {{
      "condition": "<diagnosis name>",
      "icd10": "<ICD-10 code>",
      "probability": <0.0-1.0>,
      "supporting_evidence": ["<imaging feature>", "<lab finding>", "<history factor>"],
      "against_evidence": ["<any data that argues against this diagnosis>"]
    }}
  ],
  "fusion_summary": "<2-3 sentence clinical synthesis explaining the top diagnosis>"
}}

Rules:
- fused_ddx sorted by probability descending, max 5 entries.
- Every entry must cite specific evidence from the data above.
- If CXR and labs conflict, note the conflict in against_evidence."""


## Cell 8 — Algorithm Functions

**Key points:**
- **Interview** — `run_intake_interview` runs the structured Q&A loop with MedGemma
- **Report writing** — `write_clinical_report` generates markdown clinical report from interview + EHR
- **AI triage** — `run_ai_triage` calls MedGemma and returns structured JSON; validates department list
- **CXR analysis** — `run_cxr_analysis` handles single and longitudinal (multi-image) modes with RAG
- **Lab parsing + DDx fusion** — `parse_lab_text` + `run_lab_ddx_fusion` produce ranked unified diagnosis
- **Drug safety** — `run_drug_safety_check` covers allergies, cross-reactivity, DDI, contraindications, PGx
- CLI helpers (`human_verify_triage`, `ask_proposed_drug`) are used by the LangGraph CLI runner (Cell 10)


In [7]:

# ============================================================
# 8. Algorithm Functions  (Mod1 -> Mod2 -> Mod3 -> Mod4 logic)
# ============================================================

# ── 8-A  Mod1 – Interview loop ───────────────────────────────

def run_intake_interview(patient_name: str, ehr_summary: str, model) -> dict:
    """Run the clinical intake text interview. Returns {question: answer} dict."""
    instructions = prompt_intake_system(patient_name, ehr_summary)
    dialog = [
        {"role": "system", "content": [{"type": "text", "text": instructions}]},
        {"role": "user",   "content": [{"type": "text", "text": "start interview"}]},
    ]
    conversation, q_count = {}, 0
    MAX_Q = 20
    while q_count < MAX_Q:
        assistant_text = ask_model(model, dialog)
        dialog.append({"role": "assistant", "content": [{"type": "text", "text": assistant_text}]})
        q_count += 1
        if "End interview." in assistant_text:
            break
        user_text = input("\n  Patient: ").strip()
        conversation[assistant_text.strip()] = user_text
        dialog.append({"role": "user", "content": [{"type": "text", "text": user_text}]})
    if q_count >= MAX_Q:
        warnings.warn("Max interview questions reached. Ending interview.")
    return conversation


# ── 8-B  Mod1 – Report writer ────────────────────────────────

def write_clinical_report(model, ehr_summary: str, interview_dict: dict) -> str:
    """Generate a markdown clinical intake report from an interview + EHR."""
    messages = [
        {"role": "system", "content": [{"type": "text", "text": prompt_report_writer_system(ehr_summary)}]},
        {"role": "user",   "content": [{"type": "text", "text": prompt_report_writer_user(interview_dict, ehr_summary)}]},
    ]
    raw = ask_model(model, messages).strip()
    # Strip code fences if model wraps output
    match = re.match(r'^\s*```(?:markdown)?\s*(.*?)\s*```\s*$', raw, re.DOTALL | re.IGNORECASE)
    return match.group(1).strip() if match else raw


# ── 8-C  Mod2 – AI triage call ──────────────────────────────

def run_ai_triage(model, patient_name: str, clinical_history: str,
                  report_summary: str, departments: List[str]) -> dict:
    """Call MedGemma triage and return parsed dict. Falls back gracefully."""
    imaging_tests = ["X-Ray", "CT Scan", "MRI", "Ultrasound", "Echocardiography"]
    lab_tests     = ["CBC", "Lipid Panel", "Blood Glucose", "Liver Function", "TSH", "Troponin"]
    messages = [
        {"role": "system", "content": [{"type": "text",
            "text": prompt_triage_system(departments, imaging_tests, lab_tests)}]},
        {"role": "user",   "content": [{"type": "text",
            "text": prompt_triage_user(patient_name, clinical_history, report_summary)}]},
    ]
    raw = ask_model(model, messages)
    data = safe_parse_json(raw)
    if not data:
        warnings.warn(f"Triage JSON parse failed. Raw output:\n{raw[:300]}")
        return {"triage_priority": "STABLE", "departments": departments[:1] if departments else [],
                "recommended_imaging": [], "recommended_labs": [],
                "clinical_summary": "Auto-triage failed; defaulted.", "_parse_error": True}
    # Validate departments against known list
    raw_depts  = data.get("departments", [])
    clean_depts = [d for d in raw_depts if d in departments]
    if not clean_depts:
        clean_depts = [departments[0]] if departments else []
    data["departments"] = clean_depts
    return data


# ── 8-D  Mod2 – Human verification (CLI) ────────────────────

def human_verify_triage(triage_result: dict, departments: List[str]) -> dict:
    """Present triage summary to user; allow approve or department override."""
    print("\n" + "-"*55)
    print("  MOD2 -- TRIAGE REVIEW (Human Verification)")
    print("-"*55)
    print(f"  Priority         : {triage_result.get('triage_priority')}")
    print(f"  Departments      : {triage_result.get('departments')}")
    print(f"  Clinical summary : {triage_result.get('clinical_summary')}")
    imaging = triage_result.get("recommended_imaging", [])
    if imaging:
        print(f"  Imaging suggested: {[i.get('name') for i in imaging]}")
    print("-"*55)
    decision = input("  Approve triage? (yes / no): ").strip().lower()
    if decision != "yes":
        print(f"\n  Available departments: {departments}")
        override = input("  Enter correct department name: ").strip()
        if override in departments:
            triage_result["departments"] = [override]
            print(f"  -> Department overridden to: {override}")
        else:
            warnings.warn(f"'{override}' not in registry. Keeping AI suggestion.")
    triage_result["human_approved"] = (decision == "yes")
    return triage_result

def human_verify_slot(slot: Dict) -> bool:
    """Show the proposed appointment slot; ask user to confirm or reject."""
    print("\n" + "-"*55)
    print("  MOD2 -- APPOINTMENT SLOT CONFIRMATION")
    print("-"*55)
    print(f"  Doctor       : {slot.get('doctor_name')}")
    print(f"  Hospital     : {slot.get('hospital_name')}")
    print(f"  Department   : {slot.get('department')}")
    print(f"  Date / Time  : {slot.get('time')}")
    print(f"  Slot ID      : {slot.get('slot_id')}")
    print("-"*55)
    answer = input("  Confirm this booking? (yes / no): ").strip().lower()
    return answer == "yes"


# ── 8-E  Mod3 – CXR image analysis ──────────────────────────

def _pad_image_to_square(img: Image.Image) -> Image.Image:
    """Pad a PIL image to a square canvas, handling grayscale/RGBA."""
    arr = sk_util.img_as_ubyte(np.array(img))
    if arr.ndim < 3:
        arr = sk_color.gray2rgb(arr)
    if arr.shape[2] == 4:
        arr = sk_color.rgba2rgb(arr)
    h, w = arr.shape[:2]
    if h == w:
        return Image.fromarray(arr.astype(np.uint8))
    if h < w:
        d = w - h
        arr = np.pad(arr, ((d // 2, d - d // 2), (0, 0), (0, 0)))
    else:
        d = h - w
        arr = np.pad(arr, ((0, 0), (d // 2, d - d // 2), (0, 0)))
    return Image.fromarray(arr.astype(np.uint8))

def _load_cxr_image(path_str: str) -> Image.Image:
    p = Path(path_str).expanduser()
    if not p.exists():
        raise FileNotFoundError(f"Image not found: {p}")
    img = Image.open(p)
    return _pad_image_to_square(img)

def ask_cxr_paths(mode: str) -> List[str]:
    """Prompt the user for CXR image path(s)."""
    if mode == "longitudinal":
        print("\n  Longitudinal analysis -- provide two image paths.")
        prev    = input("  Path to PREVIOUS CXR image: ").strip()
        current = input("  Path to CURRENT  CXR image: ").strip()
        return [prev, current]
    else:
        path = input("\n  Path to CXR image: ").strip()
        return [path]

def run_cxr_analysis(image_paths: List[str], mode: str, model,
                     anatomy: str = "Thorax", instruction: str = "") -> str:
    """Run CXR image diagnosis (single or longitudinal) using MedGemma + RAG."""
    try:
        images = [_load_cxr_image(p) for p in image_paths if p]
    except FileNotFoundError as e:
        return f"[ERROR] {e}"
    if not images:
        return "[ERROR] No valid images provided."

    if mode == "longitudinal" and len(images) >= 2:
        prompt_text = prompt_longitudinal_cxr(len(images))
        content = [{"type": "image", "image": img} for img in images]
        content.append({"type": "text", "text": prompt_text})
    else:
        # Single image + RAG
        query    = f"CXR {anatomy} image findings protocol context"
        retrieved = search_cxr_context(query)   # graceful fallback -- always returns list
        prompt_text = prompt_cxr_diagnosis(retrieved, anatomy, instruction)
        content = [{"type": "image", "image": images[0]}, {"type": "text", "text": prompt_text}]

    messages = [{"role": "user", "content": content}]
    return ask_model(model, messages)

def parse_cxr_structured_output(raw: str) -> dict:
    """
    Parse the structured JSON output from run_cxr_analysis.
    Falls back to {findings: raw, regions_of_interest: [], ddx: []} on failure.
    """
    parsed = safe_parse_json(raw)
    if parsed and isinstance(parsed, dict) and "findings" in parsed:
        parsed.setdefault("regions_of_interest", [])
        parsed.setdefault("ddx", [])
        return parsed
    # Fallback: treat entire response as free-text findings
    return {"findings": raw, "regions_of_interest": [], "ddx": []}


# ── 8-F  Mod3 – Lab result parsing and DDx fusion ────────────

def parse_lab_text(raw_lab_text: str, model) -> dict:
    """
    Send raw lab text to MedGemma for structured extraction.
    Returns {labs: [{name, value, unit, reference_range, status}]}
    """
    if not raw_lab_text or not raw_lab_text.strip():
        return {"labs": []}
    messages = [
        {"role": "user", "content": [{"type": "text",
            "text": prompt_lab_parser(raw_lab_text)}]},
    ]
    raw = ask_model(model, messages)
    parsed = safe_parse_json(raw)
    if parsed and "labs" in parsed:
        return parsed
    return {"labs": []}

def run_lab_ddx_fusion(model, cxr_ddx: list, lab_dict: dict, ehr_summary: str) -> dict:
    """
    Fuse CXR DDx with lab results via MedGemma.
    Returns {fused_ddx: [...], fusion_summary: "..."}
    """
    messages = [
        {"role": "user", "content": [{"type": "text",
            "text": prompt_lab_ddx_fusion(lab_dict, cxr_ddx, ehr_summary)}]},
    ]
    raw = ask_model(model, messages)
    parsed = safe_parse_json(raw)
    if parsed and "fused_ddx" in parsed:
        return parsed
    # Fallback: pass CXR DDx through unchanged
    return {"fused_ddx": cxr_ddx, "fusion_summary": "Lab fusion parse failed; CXR DDx used."}


# ── 8-G  Mod4 – Drug safety check ────────────────────────────

def _load_drug_kb() -> dict:
    global _DRUG_KB
    if _DRUG_KB is not None:
        return _DRUG_KB
    if not DRUG_KNOWLEDGE_JSON.exists():
        raise FileNotFoundError(f"Drug knowledge JSON not found: {DRUG_KNOWLEDGE_JSON}")
    with open(DRUG_KNOWLEDGE_JSON) as f:
        _DRUG_KB = json.load(f)
    return _DRUG_KB

def _norm(s: str) -> str:
    return str(s).lower().strip()

def _drug_matches(proposed: str, drug_list: List[str]) -> bool:
    """
    Fuzzy drug name match using difflib.SequenceMatcher.
    Catches variants like 'metformin HCl' vs 'Metformin', 'Aspirin 81mg' vs 'Aspirin'.
    Falls back to exact _norm comparison as baseline.
    """
    p = _norm(proposed)
    for d in drug_list:
        d_norm = _norm(d)
        # Exact match
        if p == d_norm:
            return True
        # One name is a substring of the other (handles "metformin HCl" / "Metformin")
        if p in d_norm or d_norm in p:
            return True
        # Fuzzy ratio (catches typos and brand/generic variations)
        ratio = difflib.SequenceMatcher(None, p, d_norm).ratio()
        if ratio >= FUZZY_MATCH_THRESHOLD:
            return True
    return False

def _severity_rank(s: str) -> int:
    return {"CRITICAL": 4, "HIGH": 3, "MODERATE": 2, "LOW": 1, "NONE": 0}.get(s.upper(), 0)

def _map_severity(s: str) -> str:
    return {"SEVERE": "CRITICAL", "CRITICAL": "CRITICAL", "HIGH": "HIGH",
            "MODERATE": "MODERATE", "LOW": "LOW", "NONE": "NONE"}.get(str(s).upper(), "HIGH")

def _check_allergies(proposed: str, patient_allergies: List[dict]) -> List[dict]:
    kb, violations = _load_drug_kb(), []
    p_norm = _norm(proposed)
    # Direct allergy
    for a in patient_allergies:
        if _drug_matches(proposed, [a.get("substance", "")]):
            violations.append({
                "type": "Allergy / Cross-Reactivity", "action": "BLOCK",
                "severity": _map_severity(a.get("severity", "HIGH")),
                "detail": f"Patient has documented allergy to {proposed} (reaction: {a.get('reaction','Unknown')}).",
                "recommendation": "Do not prescribe. Select an alternative from a different class.",
            })
    # Cross-reactivity
    for rule in kb.get("allergy_cross_reactivity", []):
        allergen = rule.get("primary_allergen", "")
        if not any(_drug_matches(a.get("substance",""), [allergen]) for a in patient_allergies):
            continue
        if _drug_matches(proposed, rule.get("cross_reactive_drugs", [])):
            allergy_sev  = next((a.get("severity","MODERATE") for a in patient_allergies
                                 if _drug_matches(a.get("substance",""), [allergen])), "MODERATE")
            action = "BLOCK" if allergy_sev.upper() in ("CRITICAL","HIGH","SEVERE") else "WARN"
            violations.append({
                "type": "Allergy / Cross-Reactivity", "action": action,
                "severity": _map_severity(rule.get("severity","MODERATE")),
                "detail": (f"Cross-reactivity risk: patient allergic to {allergen}; "
                           f"{proposed} shares structural similarity "
                           f"(rate: {rule.get('cross_reactivity_rate','?')})."),
                "recommendation": rule.get("note", "Use with caution or avoid."),
            })
    return violations

def _check_ddi(proposed: str, patient_meds: List[dict]) -> List[dict]:
    kb, violations = _load_drug_kb(), []
    med_names = [m.get("name","") for m in patient_meds]
    for rule in kb.get("drug_drug_interactions", []):
        a_mem = rule.get("drug_a_members", [])
        b_mem = rule.get("drug_b_members", [])
        proposed_is_a = _drug_matches(proposed, a_mem)
        proposed_is_b = _drug_matches(proposed, b_mem)
        if not proposed_is_a and not proposed_is_b:
            continue
        other_members = b_mem if proposed_is_a else a_mem
        for med in med_names:
            if _drug_matches(med, other_members):
                violations.append({
                    "type": "Drug-Drug Interaction", "action": rule.get("action","WARN"),
                    "severity": rule.get("severity","MODERATE"),
                    "detail": (f"{proposed} interacts with {med} "
                               f"(rule: {rule.get('id','?')})."),
                    "mechanism":      rule.get("mechanism",""),
                    "adverse_effect": rule.get("adverse_effect",""),
                    "recommendation": rule.get("recommendation",""),
                    "interacting_with": med,
                })
    return violations

def _check_contraindications(proposed: str, conditions: List[str], renal: dict) -> List[dict]:
    kb, violations = _load_drug_kb(), []
    cond_norm = [_norm(c) for c in conditions]
    for rule in kb.get("drug_disease_contraindications", []):
        drugs = rule.get("drug", []) + rule.get("drug_class", [])
        if not _drug_matches(proposed, drugs):
            continue
        condition = rule.get("condition","")
        if "egfr_threshold" in rule:
            egfr = renal.get("egfr", 999)
            if egfr < rule["egfr_threshold"]:
                violations.append({
                    "type": "Renal Dose Adjustment", "action": rule.get("action","BLOCK"),
                    "severity": rule.get("severity","HIGH"),
                    "detail": (f"{proposed} contraindicated: patient eGFR {egfr} "
                               f"(threshold {rule['egfr_threshold']})."),
                    "recommendation": rule.get("recommendation",""),
                })
        elif _norm(condition) in cond_norm:
            violations.append({
                "type": "Drug-Disease Contraindication", "action": rule.get("action","BLOCK"),
                "severity": rule.get("severity","MODERATE"),
                "detail":   f"{proposed} contraindicated with '{condition}'.",
                "recommendation": rule.get("recommendation",""),
            })
    return violations

def _check_pharmacogenomics(proposed: str, gene_profile: dict) -> List[dict]:
    """
    Check proposed drug against the pharmacogenomics KB section.
    gene_profile: {gene_name: phenotype_string} e.g. {"CYP2C19": "Poor Metabolizer"}
    Returns list of violation dicts.
    """
    if not gene_profile:
        return []
    kb = _load_drug_kb()
    violations = []
    for rule in kb.get("pharmacogenomics_interactions", []):
        gene         = rule.get("gene", "")
        risk_pheno   = rule.get("risk_phenotype", "")
        affected     = rule.get("affected_drugs", [])
        patient_pheno = gene_profile.get(gene, "")

        if not patient_pheno:
            continue
        # Check if patient phenotype matches this risk
        if _norm(patient_pheno) != _norm(risk_pheno):
            continue
        if not _drug_matches(proposed, affected):
            continue

        severity = rule.get("severity", "HIGH")
        action   = "BLOCK" if severity == "CRITICAL" else "WARN"
        violations.append({
            "type":           "Pharmacogenomics Interaction",
            "action":         action,
            "severity":       severity,
            "gene":           gene,
            "patient_phenotype": patient_pheno,
            "detail": (f"{proposed} is affected by {gene} {patient_pheno} phenotype. "
                       f"{rule.get('mechanism','')}"),
            "recommendation": rule.get("recommendation", ""),
            "evidence_source": rule.get("evidence_source", ""),
        })
    return violations

def ask_proposed_drug() -> tuple:
    """Prompt clinician for drug details at the terminal. Returns (drug, dosage, indication)."""
    print("\n  MOD4 -- PRESCRIPTION SAFETY CHECK")
    print("-"*55)
    drug       = input("  Proposed drug name     : ").strip()
    dosage     = input("  Dosage (e.g. 500mg BD) : ").strip()
    indication = input("  Indication / reason    : ").strip()
    return drug, dosage, indication

def run_drug_safety_check(proposed_drug: str, dosage: str, indication: str,
                          patient_context: dict) -> dict:
    """
    Full drug safety check: allergies + DDI + contraindications + pharmacogenomics.
    patient_context keys: allergies, current_medications, conditions, renal_function, gene_profile
    """
    allergies    = patient_context.get("allergies", [])
    meds         = patient_context.get("current_medications", [])
    conditions   = patient_context.get("conditions", [])
    renal        = patient_context.get("renal_function", {"egfr": 999, "status": "NORMAL"})
    gene_profile = patient_context.get("gene_profile", {})

    all_violations = (
        _check_allergies(proposed_drug, allergies)
        + _check_ddi(proposed_drug, meds)
        + _check_contraindications(proposed_drug, conditions, renal)
        + _check_pharmacogenomics(proposed_drug, gene_profile)
    )

    blocks   = [v for v in all_violations if v.get("action") == "BLOCK"]
    warnings_list = [v for v in all_violations if v.get("action") != "BLOCK"]

    if blocks:
        status      = "BLOCK"
        alert_level = max(blocks,        key=lambda v: _severity_rank(v.get("severity","LOW"))).get("severity","HIGH")
    elif warnings_list:
        status      = "PASS"
        alert_level = max(warnings_list, key=lambda v: _severity_rank(v.get("severity","LOW"))).get("severity","LOW")
    else:
        status, alert_level = "PASS", "NONE"

    pgx_violations = [v for v in all_violations if v.get("type") == "Pharmacogenomics Interaction"]

    return {
        "proposed_drug": proposed_drug, "dosage": dosage, "indication": indication,
        "status": status, "alert_level": alert_level,
        "blocks": blocks, "warnings": warnings_list,
        "pharmacogenomics": pgx_violations,
    }


## Cell 9 — LangGraph Workflow

**Key points:**
- `RouterState` TypedDict carries all pipeline data between nodes
- **5 nodes:** `node_intake` → `node_triage` → `[node_cxr →]` `node_drug_safety` → `node_booking`
- **Conditional routing** — `route_after_triage` sends to `node_cxr` only when a CXR department is selected
- Uses `MemorySaver` checkpoint so pipeline state is recoverable across cell restarts
- The compiled `PIPELINE_COMPILED` graph is available globally after this cell runs


In [8]:

# ============================================================
# 9. LangGraph Workflow  (RouterState + 5 nodes + compiled graph)
# ============================================================

# ── 9-A  RouterState ─────────────────────────────────────────

class RouterState(TypedDict):
    # Patient identity
    patient_id      : str
    patient_details : dict
    last_visit      : int            # 0 = new, 1 = returning
    # Mod1 – Intake
    ehr_summary     : str
    interview       : dict
    report_summary  : str
    # Mod2 – Triage
    triage_result        : dict
    triage_approved      : bool
    selected_departments : List[str]
    # Mod3 – CXR + structured output
    cxr_triggered       : bool
    cxr_mode            : str
    cxr_paths           : List[str]
    cxr_findings        : str                   # raw model output (kept for backward-compat)
    cxr_structured      : dict                   # {findings, regions_of_interest, ddx}
    # Mod3 – Lab results
    lab_results_text    : str                    # raw text entered by clinician
    lab_results_parsed  : dict                   # {labs: [{name, value, unit, ...}]}
    lab_fusion_result   : dict                   # {fused_ddx, fusion_summary}
    # Mod4 – Drug safety
    proposed_drug       : str
    proposed_dosage     : str
    drug_indication     : str
    drug_safety_result  : dict
    # Booking
    available_slots : List[dict]
    selected_slot   : Optional[dict]
    appt_id         : str
    booking_status  : str
    # FHIR bundle output path
    fhir_bundle_path : str
    # Audit
    error_log      : Annotated[list, operator.add]
    routing_notes  : Annotated[list, operator.add]


# ── 9-B  Node functions ──────────────────────────────────────

def node_intake(state: RouterState) -> dict:
    """Mod1: Intake interview + clinical report."""
    model   = get_medgemma()
    details = state.get("patient_details", {})
    name    = details.get("name", "Patient")

    # ehr_summary pre-computed in run_pipeline before graph entry
    ehr_summary    = state.get("ehr_summary", "")
    interview      = run_intake_interview(name, ehr_summary, model)
    report_summary = write_clinical_report(model, ehr_summary, interview)

    return {
        "ehr_summary"    : ehr_summary,
        "interview"      : interview,
        "report_summary" : report_summary,
        "routing_notes"  : ["Mod1 intake complete"],
    }


def node_triage(state: RouterState) -> dict:
    """Mod2: AI triage + dual human verification."""
    model   = get_medgemma()
    name    = state.get("patient_details", {}).get("name", "Patient")
    history = state.get("ehr_summary", "")
    report  = state.get("report_summary", "")
    depts   = get_all_departments()

    triage_raw   = run_ai_triage(model, name, history, report, depts)
    triage_final = human_verify_triage(triage_raw, depts)

    selected_depts = triage_final.get("departments", [])
    cxr_depts      = list(DEPARTMENT_TO_MODALITIES.keys())
    cxr_triggered  = any(d in cxr_depts for d in selected_depts)

    return {
        "triage_result"        : triage_final,
        "triage_approved"      : triage_final.get("human_approved", False),
        "selected_departments" : selected_depts,
        "cxr_triggered"        : cxr_triggered,
        "routing_notes"        : [f"Triage -> depts={selected_depts}, CXR={cxr_triggered}"],
    }


def node_cxr(state: RouterState) -> dict:
    """Mod3: CXR image analysis (single or longitudinal) + lab parsing + DDx fusion."""
    model = get_medgemma()
    pid   = state["patient_id"]
    mode  = detect_cxr_mode(pid)

    paths      = ask_cxr_paths(mode)
    raw_out    = run_cxr_analysis(paths, mode, model)
    cxr_struct = parse_cxr_structured_output(raw_out)

    ddx = cxr_struct.get("ddx", [])

    # Lab result input
    print("\n  LAB RESULTS (optional) -- paste raw lab text or press Enter to skip:")
    lab_text = input("  > ").strip()
    lab_parsed = {"labs": []}
    lab_fusion = {"fused_ddx": ddx, "fusion_summary": "No lab data provided."}
    if lab_text:
        lab_parsed = parse_lab_text(lab_text, model)
        lab_fusion = run_lab_ddx_fusion(model, ddx, lab_parsed, state.get("ehr_summary", ""))

    return {
        "cxr_mode"         : mode,
        "cxr_paths"        : paths,
        "cxr_findings"     : raw_out,
        "cxr_structured"   : cxr_struct,
        "lab_results_text" : lab_text,
        "lab_results_parsed": lab_parsed,
        "lab_fusion_result": lab_fusion,
        "routing_notes"    : [f"Mod3 CXR complete (mode={mode}), labs={'yes' if lab_text else 'skipped'}"],
    }


def node_drug_safety(state: RouterState) -> dict:
    """Mod4: Drug safety check including pharmacogenomics."""
    drug, dosage, indication = ask_proposed_drug()

    details = state.get("patient_details", {})
    ctx = {
        "allergies"           : details.get("allergies", []),
        "current_medications" : details.get("current_medications", []),
        "conditions"          : details.get("conditions", []),
        "renal_function"      : details.get("renal_function", {"egfr": 999, "status": "NORMAL"}),
        "gene_profile"        : details.get("gene_profile", {}),
    }
    result = run_drug_safety_check(drug, dosage, indication, ctx)

    return {
        "proposed_drug"     : drug,
        "proposed_dosage"   : dosage,
        "drug_indication"   : indication,
        "drug_safety_result": result,
        "routing_notes"     : [f"Mod4 drug-safety status={result['status']}"],
    }


def node_booking(state: RouterState) -> dict:
    """Find a free slot, human-confirm, then book."""
    selected_depts = state.get("selected_departments", [])
    dept   = selected_depts[0] if selected_depts else None
    slots  = get_free_slots_from_registry(dept) if dept else []
    if not slots:
        fallback = get_any_free_slot()
        slots    = [fallback] if fallback else []

    if not slots:
        return {
            "booking_status": "NO_SLOTS",
            "appt_id"       : "",
            "error_log"     : ["No available slots found in registry."],
        }

    slot    = slots[0]
    pid     = state["patient_id"]
    confirm = human_verify_slot(slot)

    if not confirm:
        return {
            "booking_status": "DECLINED",
            "selected_slot" : slot,
            "appt_id"       : "",
            "routing_notes" : ["Booking declined by user."],
        }

    appt_id = confirm_booking(pid, slot)
    print(f"\n  Appointment confirmed -- ID: {appt_id}")
    return {
        "booking_status" : "CONFIRMED",
        "selected_slot"  : slot,
        "appt_id"        : appt_id,
        "available_slots": slots,
        "routing_notes"  : [f"Booking confirmed: {appt_id}"],
    }


# ── 9-C  Conditional routing edge (after triage) ────────────

def route_after_triage(state: RouterState) -> str:
    return "node_cxr" if state.get("cxr_triggered") else "node_drug_safety"


# ── 9-D  Build & compile graph ───────────────────────────────

def build_pipeline_graph() -> StateGraph:
    g = StateGraph(RouterState)
    g.add_node("node_intake",      node_intake)
    g.add_node("node_triage",      node_triage)
    g.add_node("node_cxr",         node_cxr)
    g.add_node("node_drug_safety", node_drug_safety)
    g.add_node("node_booking",     node_booking)

    g.set_entry_point("node_intake")
    g.add_edge("node_intake", "node_triage")
    g.add_conditional_edges(
        "node_triage", route_after_triage,
        {"node_cxr": "node_cxr", "node_drug_safety": "node_drug_safety"},
    )
    g.add_edge("node_cxr",         "node_drug_safety")
    g.add_edge("node_drug_safety", "node_booking")
    g.add_edge("node_booking",     END)
    return g


PIPELINE_GRAPH    = build_pipeline_graph()
PIPELINE_CHECKPT  = MemorySaver()
PIPELINE_COMPILED = PIPELINE_GRAPH.compile(checkpointer=PIPELINE_CHECKPT)


## Cell 10 — CLI Pipeline Runner

**Key points:**
- `run_pipeline()` — full end-to-end CLI execution; prompts for patient phone then runs all nodes
- Calls `preload_pipeline()` implicitly by loading MedGemma and the vector store before streaming
- `_print_clinical_dashboard()` — renders a structured summary table to the console after completion
- Saves: **session JSON** `+` **lab values JSON** `+` **FHIR R4 bundle** per visit
- Run `run_pipeline()` here for CLI/terminal usage; use `launch_gradio_ui()` (Cell 11) for the web UI


In [9]:

# ============================================================
# 10. Orchestration Runner  --  run_pipeline()
# ============================================================

def _print_clinical_dashboard(s: dict) -> None:
    SEP = '=' * 68
    print(f'\n{SEP}')
    print('  CLINICAL PIPELINE SUMMARY')
    print(SEP)
    info = s.get('patient_details', {})
    print(f"  Patient      : {info.get('name','--')}  |  ID: {s.get('patient_id','--')}")
    print(f"  Age / Sex    : {info.get('age','--')} / {info.get('sex','--')}")
    pgx = info.get('gene_profile', {})
    if pgx:
        print(f"  Gene profile : {pgx}")

    # MOD2 Triage
    tr = s.get('triage_result', {})
    print('\n  -- MOD2 TRIAGE --')
    print(f"  Priority       : {tr.get('triage_priority','--')}")
    print(f"  Departments    : {tr.get('departments','--')}")
    print(f"  Human approved : {tr.get('human_approved','--')}")
    cs = tr.get('clinical_summary', '')
    if cs:
        print(f"  Summary        : {cs[:150]}{'...' if len(cs)>150 else ''}")
    img_rec = tr.get('recommended_imaging', [])
    if img_rec:
        print(f"  Imaging orders : {[i.get('name') for i in img_rec]}")
    lab_rec = tr.get('recommended_labs', [])
    if lab_rec:
        print(f"  Lab orders     : {[l.get('name') for l in lab_rec]}")

    # MOD3 CXR + DDx + ROI
    if s.get('cxr_triggered'):
        print(f"\n  -- MOD3 CXR  (mode: {s.get('cxr_mode','?')}) --")
        cxr_struct = s.get('cxr_structured', {})
        findings = cxr_struct.get('findings') or s.get('cxr_findings', '')
        print(f"  Findings : {findings[:300]}{'...' if len(findings)>300 else ''}")

        rois = cxr_struct.get('regions_of_interest', [])
        if rois:
            print(f"  Regions of interest ({len(rois)}):")
            for roi in rois:
                change_tag = f"  [{roi.get('change','')}]" if roi.get('change') else ''
                print(f"    - {roi.get('label','')} @ {roi.get('location','')} ({roi.get('size','')}){change_tag}")

        # Fused DDx (prefer lab fusion if available)
        fusion = s.get('lab_fusion_result', {})
        ddx_list = fusion.get('fused_ddx') or cxr_struct.get('ddx', [])
        if ddx_list:
            print(f"\n  Differential Diagnoses:")
            print(f"  {'Rank':<5} {'Condition':<35} {'ICD-10':<10} {'Prob':>6}")
            print(f"  {'----':<5} {'---------':<35} {'------':<10} {'----':>6}")
            for i, d in enumerate(ddx_list[:5], 1):
                prob = d.get('probability', 0)
                prob_str = f"{prob:.0%}" if isinstance(prob, (int, float)) else str(prob)
                print(f"  {i:<5} {d.get('condition',''):<35} {d.get('icd10',''):<10} {prob_str:>6}")
        fusion_summary = fusion.get('fusion_summary', '')
        if fusion_summary and fusion_summary != 'No lab data provided.':
            print(f"\n  Fusion: {fusion_summary[:250]}")

        labs = s.get('lab_results_parsed', {}).get('labs', [])
        if labs:
            print(f"\n  Lab Results ({len(labs)} values):")
            for lab in labs[:8]:
                status_sym = {'high': 'H', 'low': 'L', 'critical': '!'}.get(
                    lab.get('status','normal').lower(), ' ')
                print(f"    [{status_sym}] {lab.get('name',''):<30} {lab.get('value','')} {lab.get('unit',''):<10}  ref: {lab.get('reference_range','')}")

    # MOD4 Drug Safety
    dr = s.get('drug_safety_result', {})
    if dr:
        print('\n  -- MOD4 DRUG SAFETY --')
        print(f"  Drug      : {dr.get('proposed_drug','--')}  |  {dr.get('dosage','')}")
        print(f"  Status    : {dr.get('status','--')}  (alert: {dr.get('alert_level','--')})")
        if dr.get('blocks'):
            print(f"  BLOCKED   : {len(dr['blocks'])} violation(s)")
            for b in dr['blocks']:
                print(f"    [{b.get('severity','')}] {b.get('detail','')[:100]}")
        if dr.get('warnings'):
            print(f"  Warnings  : {len(dr['warnings'])}")
        pgx_flags = dr.get('pharmacogenomics', [])
        if pgx_flags:
            print(f"  PGx flags : {len(pgx_flags)}")
            for p in pgx_flags:
                print(f"    [{p.get('gene','')} / {p.get('patient_phenotype','')}] {p.get('detail','')[:100]}")

    # Appointment
    print('\n  -- APPOINTMENT --')
    print(f"  Status    : {s.get('booking_status','--')}")
    if s.get('appt_id'):
        print(f"  Appt ID   : {s.get('appt_id')}")
    sl = s.get('selected_slot')
    if sl:
        print(f"  Slot      : {sl.get('doctor_name','--')} @ {sl.get('time','--')}  ({sl.get('hospital_name','--')})")

    # FHIR bundle
    if s.get('fhir_bundle_path'):
        print(f"\n  FHIR Bundle: {s.get('fhir_bundle_path')}")

    errs = s.get('error_log', [])
    if errs:
        print('\n  -- ERRORS --')
        for e in errs:
            print(f'  x {e}')
    print(f'\n{SEP}\n')


def run_pipeline() -> dict:
    """Full clinical pipeline entry point."""
    print('\n  Loading MedGemma ...')
    get_medgemma()
    print('  Loading embedding model ...')
    get_embed_model()
    print('  Opening CXR vector store ...')
    open_or_create_lancedb()

    print('\n' + '-'*68)
    print('  PATIENT IDENTIFICATION')
    print('-'*68)
    phone = input('  Enter patient mobile number: ').strip()
    patient_details, last_visit, ehr_records, _df = get_patient_details(phone)
    model = get_medgemma()
    name  = patient_details.get("name", "Patient")
    if ehr_records != "There is No Past Records":
        print("  Summarising past EHR records ...")
        ehr_summary = get_ehr_summary(name, ehr_records, model)
        print("  EHR summary ready.")
    else:
        ehr_summary = ""
        print("  No prior EHR records found.")
    pid  = phone
    name = patient_details.get('name', 'Patient')
    print(f'  Patient: {name}  (last_visit={last_visit})')
    pgx = patient_details.get('gene_profile', {})
    if pgx:
        print(f'  Gene profile loaded: {pgx}')

    initial_state: RouterState = {
        'patient_id'          : pid,
        'patient_details'     : patient_details,
        'last_visit'          : last_visit,
        'ehr_summary'         : ehr_summary,
        'interview'           : {},
        'report_summary'      : '',
        'triage_result'       : {},
        'triage_approved'     : False,
        'selected_departments': [],
        'cxr_triggered'       : False,
        'cxr_mode'            : 'single',
        'cxr_paths'           : [],
        'cxr_findings'        : '',
        'cxr_structured'      : {},
        'lab_results_text'    : '',
        'lab_results_parsed'  : {},
        'lab_fusion_result'   : {},
        'proposed_drug'       : '',
        'proposed_dosage'     : '',
        'drug_indication'     : '',
        'drug_safety_result'  : {},
        'available_slots'     : [],
        'selected_slot'       : None,
        'appt_id'             : '',
        'booking_status'      : '',
        'fhir_bundle_path'    : '',
        'error_log'           : [],
        'routing_notes'       : [],
    }

    thread_cfg  = {'configurable': {'thread_id': f'session-{pid}'}}
    final_state = initial_state.copy()
    print('\n  Starting pipeline ...\n')
    for chunk in PIPELINE_COMPILED.stream(initial_state, config=thread_cfg, stream_mode='values'):
        final_state = chunk

    _print_clinical_dashboard(final_state)

    fhir_path = ''
    try:
        make_directory(MOD1_CONVERSATION_DIR)
        rec = build_patient_json_record(
            patient_id      = pid,
            patient_details = patient_details,
            ehr_summary     = final_state.get('ehr_summary', ''),
            interview       = final_state.get('interview', {}),
            report_summary  = final_state.get('report_summary', ''),
            triage          = final_state.get('triage_result', {}),
            booking         = {
                'status'  : final_state.get('booking_status', ''),
                'appt_id' : final_state.get('appt_id', ''),
                'slot'    : final_state.get('selected_slot') or {},
            },
            imaging         = {
                'mode'        : final_state.get('cxr_mode', ''),
                'paths'       : final_state.get('cxr_paths', []),
                'findings'    : final_state.get('cxr_findings', ''),
                'structured'  : final_state.get('cxr_structured', {}),
                'lab_parsed'  : final_state.get('lab_results_parsed', {}),
                'lab_fusion'  : final_state.get('lab_fusion_result', {}),
            },
            drug_safety     = final_state.get('drug_safety_result', {}),
        )
        sid      = rec['meta']['session_id'][:8]
        out_path = MOD1_CONVERSATION_DIR / f'session_{pid}_{sid}.json'
        save_json(rec, str(out_path))
        print(f'  Session JSON -> {out_path}')

        # Save lab values separately
        lab_parsed = final_state.get('lab_results_parsed', {})
        if lab_parsed.get('labs'):
            make_directory(LAB_SESSIONS_DIR)
            lab_path = LAB_SESSIONS_DIR / f'{pid}_{sid}_labs.json'
            save_json(lab_parsed, str(lab_path))
            print(f'  Lab JSON     -> {lab_path}')

        # Build and save FHIR R4 bundle
        fhir_resources = []
        fhir_resources.append(build_fhir_patient_resource(patient_details))
        cxr_struct = final_state.get('cxr_structured', {})
        for ddx_entry in cxr_struct.get('ddx', []):
            fhir_resources.append(build_fhir_condition_resource(ddx_entry, pid))
        for lab in lab_parsed.get('labs', []):
            fhir_resources.append(build_fhir_observation_resource(lab, pid))
        dr = final_state.get('drug_safety_result', {})
        if dr.get('proposed_drug'):
            fhir_resources.append(build_fhir_medication_request_resource(
                dr['proposed_drug'], dr.get('dosage', ''), dr, pid))
        fhir_path = build_fhir_bundle(pid, sid, *fhir_resources)
        print(f'  FHIR Bundle  -> {fhir_path}')

    except Exception as exc:
        warnings.warn(f"Could not save session/FHIR: {exc}")

    final_state['fhir_bundle_path'] = fhir_path
    return final_state


## Cell 11 — Gradio Clinical UI

**Key points:**
- `launch_gradio_ui()` — builds and launches the 6-tab sequential UI at `http://localhost:7860`
- **Pre-loads** model + registries before the first patient if `preload=True` (default)
- **Loading indicators** on all async steps (registration, report, triage, CXR, drug check, save)
- **Routing card** (dept / doctor / queue) is visible in triage tab *before* clicking Confirm
- **Last visit summary** is shown in the registration card for returning patients
- **Follow-up date** field uses a plain textbox (any clear date format accepted)
- **Download button** appears after saving — downloads the session summary as `.txt`
- **Triage routing** — CXR-ordered patients go to ④ AI Diagnosis; others skip to ⑤ Prescription Safety


In [10]:

# ============================================================
# 11. Continuum — Sequential Clinical Pipeline UI  (Gradio)
# ============================================================
#  Tabs (in order):
#   ① Registration      — phone lookup / new patient form
#   ② Clinical Interview— chatbot Q&A → clinical report
#   ③ AI Triage         — AI assessment + routing card + confirm
#   ④ AI Diagnosis      — CXR / lab analysis (routed patients)
#   ⑤ Prescription Safety — drug safety check w/ AI Insight
#   ⑥ Complete Records  — save session, FHIR, follow-up, download
# ============================================================

import io, contextlib, tempfile, datetime, os

try:
    import gradio as gr
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.check_call([_sys.executable, "-m", "pip", "install", "gradio"])
    import gradio as gr


# ══════════════════════════════════════════════════════════════
#  PRE-LOAD  — warm models + registries before any user arrives
# ══════════════════════════════════════════════════════════════

_PRELOADED: dict = {}


def preload_pipeline() -> None:
    """Load MedGemma + all registry data once so first patient sees instant UI."""
    _PRELOADED["model"]       = get_medgemma()
    _PRELOADED["patient_df"]  = load_patient_registry()
    _PRELOADED["provider_df"] = load_provider_registry()
    _PRELOADED["departments"] = get_all_departments()


def _get_model():
    return _PRELOADED.get("model") or get_medgemma()


def _get_departments() -> list:
    return _PRELOADED.get("departments") or get_all_departments()


# ══════════════════════════════════════════════════════════════
#  PIPELINE STATE
# ══════════════════════════════════════════════════════════════

def _fresh_state() -> dict:
    return {
        # registration
        "patient_id": "", "patient_details": {}, "last_visit": 0,
        "last_visit_date": "", "last_visit_summary": "",
        "ehr_records": "There is No Past Records", "ehr_summary": "",
        # interview
        "dialog": [], "interview": {}, "q_count": 0,
        "interview_done": False, "report_summary": "",
        "appt_date": "",
        # triage
        "triage_result": {}, "triage_approved": False,
        "selected_departments": [], "cxr_triggered": False,
        "assigned_slot": {},
        # CXR / labs
        "cxr_mode": "single", "cxr_paths": [], "cxr_findings": "",
        "cxr_structured": {}, "lab_results_text": "",
        "lab_results_parsed": {}, "lab_fusion_result": {},
        "ai_insight": {},
        # drug safety
        "proposed_drug": "", "proposed_dosage": "",
        "drug_indication": "", "drug_safety_result": {},
        # records
        "session_id": "", "fhir_bundle_path": "",
        "follow_up_required": False, "follow_up_date": "",
        "saved_txt_path": "",
    }


# ══════════════════════════════════════════════════════════════
#  SHARED HELPERS
# ══════════════════════════════════════════════════════════════

def _build_routing_card(slot: dict, depts: list) -> str:
    """Build a markdown clinic-assignment table from a provider slot dict."""
    dept   = slot.get("department", ", ".join(depts)) if slot else ", ".join(depts)
    doctor = slot.get("doctor_name", "--")
    hosp   = slot.get("hospital_name", "--")
    time_  = slot.get("time", "--")
    queue  = slot.get("slot_id", "--")
    return (
        f"\n---\n#### 🏥 Clinic Assignment\n\n"
        f"| | |\n|---|---|\n"
        f"| **Department** | {dept} |\n"
        f"| **Doctor** | {doctor} |\n"
        f"| **Hospital** | {hosp} |\n"
        f"| **Appointment Time** | {time_} |\n"
        f"| **Queue Number** | `{queue}` |\n"
    )


def _build_overall_record_md(state: dict) -> str:
    """Build a complete patient record summary markdown for display after save."""
    info   = state.get("patient_details", {})
    name   = info.get("name", "--")
    today  = datetime.date.today().strftime("%d %B %Y")
    appt   = state.get("appt_date", today)
    sid    = state.get("session_id", "--")
    slot   = state.get("assigned_slot", {})

    md  = f"---\n## 📋 Complete Patient Record\n\n"
    md += f"| Field | Value |\n|---|---|\n"
    md += f"| **Patient** | {name} |\n"
    md += f"| **ID** | {state.get('patient_id','--')} |\n"
    md += f"| **Age / Sex** | {info.get('age','--')} / {info.get('sex','--')} |\n"
    md += f"| **Report Date** | {today} |\n"
    md += f"| **Appointment** | {appt} |\n"
    md += f"| **Session** | `{sid}` |\n"
    md += f"| **Prior Visits** | {state.get('last_visit',0)} |\n\n"

    # Last visit
    lvs = state.get("last_visit_summary", "")
    if lvs:
        md += f"**Last Visit Summary:**\n> {lvs[:300]}{'...' if len(lvs)>300 else ''}\n\n"

    # EHR / report
    ehr = state.get("ehr_summary", "")
    if ehr and ehr != "There is No Past Records":
        md += f"**EHR Summary:** {ehr[:250]}{'...' if len(ehr)>250 else ''}\n\n"
    rpt = state.get("report_summary", "")
    if rpt:
        md += f"**Clinical Report:** {rpt[:300]}{'...' if len(rpt)>300 else ''}\n\n"

    # Triage
    tr = state.get("triage_result", {})
    if tr:
        md += (
            f"**Triage:** `{tr.get('triage_priority','--')}` — "
            f"{', '.join(tr.get('departments',[]))} — "
            f"Approved: {'Yes' if tr.get('human_approved') else 'Overridden'}\n\n"
        )

    # AI Insight
    ai = state.get("ai_insight", {})
    if ai:
        top = ai.get("top_diagnosis", {})
        if top:
            prob = top.get("probability", 0)
            ps   = f" ({prob:.0%})" if isinstance(prob, (int, float)) else ""
            md  += f"**Top Diagnosis:** {top.get('condition','--')} `{top.get('icd10','')}`{ps}\n\n"
        fs = ai.get("fusion_summary", "")
        if fs:
            md += f"**Fusion Summary:** {fs[:200]}\n\n"

    # Drug safety
    dr = state.get("drug_safety_result", {})
    if dr.get("proposed_drug"):
        alert_raw = dr.get("alert_level")
        alert_str = str(alert_raw).upper() if alert_raw not in (None, "", "None") else "NONE"
        icon = "🚫" if dr.get("status") == "BLOCK" else "✅"
        md += f"**Prescription:** {icon} {dr['proposed_drug']} {dr.get('dosage','')} — Alert: `{alert_str}`\n\n"

    # Routing
    if slot:
        md += _build_routing_card(slot, state.get("selected_departments", []))

    # Follow-up
    if state.get("follow_up_required"):
        fu_date = state.get("follow_up_date") or "Date not specified"
        md += f"\n**Follow-Up:** Required — {fu_date}\n"

    return md


# ══════════════════════════════════════════════════════════════
#  STEP 0  —  Patient Registration
# ══════════════════════════════════════════════════════════════

def step0_find_patient(phone: str, state: dict):
    """
    Look up patient by phone.
    - Existing → card with last-visit date + summary snippet + Proceed button.
    - New       → inline registration form.
    Returns: (state, patient_card, proceed_btn, new_patient_row, status_msg)
    """
    phone = (phone or "").strip()
    if not phone:
        return (
            state,
            gr.update(value=""),
            gr.update(visible=False),
            gr.update(visible=False),
            gr.update(value="⚠️ Please enter the patient mobile number."),
        )

    reg_row     = lookup_patient_by_phone(phone)
    last_visit  = 0
    ehr_records = "There is No Past Records"

    if reg_row:
        patient_id       = reg_row.get("global_patient_id", phone)
        gene_profile_raw = reg_row.get("gene_profile", "{}")
        try:
            gene_profile = json.loads(gene_profile_raw) if isinstance(gene_profile_raw, str) else {}
        except Exception:
            gene_profile = {}

        details = {
            "patient_id":          patient_id,
            "name":                reg_row.get("patient_name", ""),
            "age":                 reg_row.get("age", ""),
            "sex":                 reg_row.get("sex", ""),
            "phone":               phone,
            "primary_dept":        reg_row.get("primary_department", ""),
            "doctor_name":         reg_row.get("primary_doctor_name", ""),
            "hospital":            reg_row.get("primary_hospital_name", ""),
            "allergies":           [],
            "current_medications": [],
            "conditions":          [],
            "renal_function":      {"egfr": 999, "status": "NORMAL"},
            "gene_profile":        gene_profile,
        }

        # ── Load EHR visit history ────────────────────────────
        last_visit_date    = ""
        last_visit_summary = ""

        if MOD1_CSV_PATH.exists():
            enc_df = pd.read_csv(MOD1_CSV_PATH)
            if "patient_id" in enc_df.columns:
                enc_df["patient_id"] = enc_df["patient_id"].astype(str)
                rows = enc_df[enc_df["patient_id"] == str(patient_id)]
            else:
                rows = pd.DataFrame()

            if not rows.empty and {"visit", "report_path"}.issubset(rows.columns):
                rows["visit"] = pd.to_numeric(rows["visit"], errors="coerce")
                rows = rows.dropna(subset=["visit"])
                if not rows.empty:
                    ehr_records = [
                        (int(v), read_txt(rp))
                        for v, rp in zip(rows["visit"].tolist(), rows["report_path"].tolist())
                        if isinstance(rp, str) and os.path.exists(rp)
                    ] or "There is No Past Records"
                    last_visit = int(rows["visit"].max())

                    # Grab last visit date + brief report snippet
                    if "date_time" in rows.columns:
                        latest = rows.sort_values("visit", ascending=False).iloc[0]
                        last_visit_date = str(latest.get("date_time", ""))[:10]
                        rp = latest.get("report_path", "")
                        if isinstance(rp, str) and os.path.exists(rp):
                            txt = read_txt(rp)
                            last_visit_summary = txt[:250] + ("…" if len(txt) > 250 else "")

        # ── Refresh state ─────────────────────────────────────
        state = _fresh_state()
        state.update({
            "patient_id":         phone,
            "patient_details":    details,
            "last_visit":         last_visit,
            "last_visit_date":    last_visit_date,
            "last_visit_summary": last_visit_summary,
            "ehr_records":        ehr_records,
        })

        pgx_line  = f"\n| **Pharmacogenomics** | {gene_profile} |" if gene_profile else ""
        visits_str = (
            f"{last_visit} prior visit(s)"
            + (f"  ·  Last: `{last_visit_date}`" if last_visit_date else "")
            if last_visit else "New patient"
        )
        card = (
            f"### ✅ Patient Found\n\n"
            f"| | |\n|---|---|\n"
            f"| **Name** | {details['name']} |\n"
            f"| **Age / Sex** | {details.get('age','--')} / {details.get('sex','--')} |\n"
            f"| **Phone** | {phone} |\n"
            f"| **Hospital** | {details.get('hospital','--')} |\n"
            f"| **Visit History** | {visits_str} |"
            f"{pgx_line}"
        )
        if last_visit_summary:
            card += (
                f"\n\n**📋 Last Visit Summary:**\n"
                f"> {last_visit_summary}"
            )

        return (
            state,
            gr.update(value=card),
            gr.update(visible=True),
            gr.update(visible=False),
            gr.update(value=""),
        )

    else:
        card = (
            f"### 🆕 New Patient\n"
            f"No record found for **{phone}**. "
            "Fill in the details below and click **Register Patient**."
        )
        return (
            state,
            gr.update(value=card),
            gr.update(visible=False),
            gr.update(visible=True),
            gr.update(value=""),
        )


def step0_register_new_patient(phone: str, name: str, age: str, sex: str, state: dict):
    """Register a brand-new patient (no input() calls)."""
    phone = (phone or "").strip()
    name  = (name  or "").strip()
    age   = (age   or "").strip()
    sex   = (sex   or "").strip()

    if not name:
        return (
            state,
            gr.update(value="⚠️ Please enter the patient name."),
            gr.update(visible=False),
            gr.update(visible=True),
        )

    details = {
        "patient_id": phone, "name": name, "age": age, "sex": sex, "phone": phone,
        "primary_dept": "", "doctor_name": "", "hospital": "",
        "allergies": [], "current_medications": [], "conditions": [],
        "renal_function": {"egfr": 999, "status": "NORMAL"}, "gene_profile": {},
    }
    state = _fresh_state()
    state["patient_id"]      = phone
    state["patient_details"] = details

    # ── Persist new patient in global registry so next visit finds them ──
    try:
        _new_row = {
            "global_patient_id":      phone,
            "patient_name":           name,
            "first_name":             name.split()[0] if name else name,
            "last_name":              " ".join(name.split()[1:]) if len(name.split()) > 1 else "",
            "date_of_birth":          "",
            "age":                    age,
            "sex":                    sex,
            "phone":                  phone,
            "email":                  "",
            "primary_department":     "",
            "primary_doctor_id":      "",
            "primary_doctor_name":    "",
            "primary_hospital_id":    "",
            "primary_hospital_name":  "",
            "city":                   "", "region": "", "country": "",
            "insurance_provider":     "", "insurance_plan": "",
            "created_at_utc":         datetime.datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S+00:00"),
        }
        if GLOBAL_PATIENT_REGISTRY_CSV.exists():
            _reg_df = pd.read_csv(GLOBAL_PATIENT_REGISTRY_CSV, dtype=str)
            # Avoid duplicate entry for same phone
            if not (_reg_df.get("phone", pd.Series(dtype=str)).str.strip() == phone.strip()).any():
                _reg_df = pd.concat([_reg_df, pd.DataFrame([_new_row])], ignore_index=True)
                _reg_df.to_csv(str(GLOBAL_PATIENT_REGISTRY_CSV), index=False)
        else:
            make_directory(GLOBAL_PATIENT_REGISTRY_CSV.parent)
            pd.DataFrame([_new_row]).to_csv(str(GLOBAL_PATIENT_REGISTRY_CSV), index=False)
    except Exception as _reg_err:
        pass  # Non-fatal: in-session state still set correctly

    card = (
        f"### ✅ Patient Registered\n\n"
        f"| | |\n|---|---|\n"
        f"| **Name** | {name} |\n"
        f"| **Age / Sex** | {age or '--'} / {sex or '--'} |\n"
        f"| **Phone** | {phone} |\n"
        f"| **Visit History** | New patient |"
    )
    return (state, gr.update(value=card), gr.update(visible=True), gr.update(visible=False))


# ══════════════════════════════════════════════════════════════
#  STEP 1  —  Clinical Interview
# ══════════════════════════════════════════════════════════════

def step1_begin_interview(state: dict):
    """Arrive at interview tab: pre-compute EHR summary and fire first question."""
    if not state.get("patient_id"):
        msg = "⚠️ Please complete Patient Registration first."
        return (
            state,
            [{"role": "assistant", "content": msg}],
            gr.update(interactive=False),
            gr.update(visible=False),
            gr.update(value=""),
        )

    model   = _get_model()
    name    = state["patient_details"].get("name", "Patient")
    ehr_rec = state["ehr_records"]

    ehr_summary = (
        get_ehr_summary(name, ehr_rec, model)
        if ehr_rec != "There is No Past Records" else ""
    )
    state["ehr_summary"] = ehr_summary

    appt_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime("%d %B %Y")
    state["appt_date"] = appt_date

    instructions = prompt_intake_system(name, ehr_summary)
    dialog = [
        {"role": "system", "content": [{"type": "text", "text": instructions}]},
        {"role": "user",   "content": [{"type": "text", "text": "start interview"}]},
    ]
    first_q = ask_model(model, dialog)
    dialog.append({"role": "assistant", "content": [{"type": "text", "text": first_q}]})
    state["dialog"]         = dialog
    state["q_count"]        = 1
    state["interview"]      = {}
    state["interview_done"] = "End interview." in first_q

    status = f"*Interviewing: **{name}**  ·  Appointment date: **{appt_date}***"
    return (
        state,
        [{"role": "assistant", "content": first_q}],
        gr.update(
            interactive=not state["interview_done"],
            placeholder="Type your answer here …"
        ),
        gr.update(visible=state["interview_done"]),
        gr.update(value=status),
    )


def step1_send_answer(user_text: str, history: list, state: dict):
    if state.get("interview_done"):
        return state, history, gr.update(value="", interactive=False), gr.update(visible=True)

    user_text = (user_text or "").strip()
    if not user_text:
        return state, history, gr.update(value=""), gr.update(visible=False)

    model  = _get_model()
    dialog = state["dialog"]

    last_q = ""
    for msg in reversed(dialog):
        if msg["role"] == "assistant":
            c = msg["content"]
            last_q = c[0]["text"] if isinstance(c, list) else c
            break

    state["interview"][last_q] = user_text
    dialog.append({"role": "user", "content": [{"type": "text", "text": user_text}]})
    history = history + [{"role": "user", "content": user_text}]
    state["q_count"] += 1

    if state["q_count"] >= 20:
        end_msg = "Thank you for your answers. I have everything I need. End interview."
        state["interview_done"] = True
        state["dialog"]         = dialog
        return (
            state,
            history + [{"role": "assistant", "content": end_msg}],
            gr.update(value="", interactive=False),
            gr.update(visible=True),
        )

    next_q = ask_model(model, dialog)
    dialog.append({"role": "assistant", "content": [{"type": "text", "text": next_q}]})
    history = history + [{"role": "assistant", "content": next_q}]
    done = "End interview." in next_q
    state["interview_done"] = done
    state["dialog"]         = dialog
    return state, history, gr.update(value="", interactive=not done), gr.update(visible=done)


def step1_generate_report(state: dict):
    """
    Generate structured clinical intake report.
    Injects patient name / age / sex / dates directly into the LLM prompt so the
    AI never outputs '- to be added' placeholders, then prepends a formatted header.
    """
    model   = _get_model()
    name    = state["patient_details"].get("name", "--")
    age     = state["patient_details"].get("age", "--")
    sex     = state["patient_details"].get("sex", "--")
    appt_dt = state.get("appt_date", datetime.date.today().strftime("%d %B %Y"))
    today   = datetime.date.today().strftime("%d %B %Y")
    ehr     = state.get("ehr_summary", "")

    # Inject patient metadata so the AI fills in all fields correctly
    patient_meta_line = (
        f"Patient: {name}  |  Age: {age}  |  Sex: {sex}  |  "
        f"Report Date: {today}  |  Appointment: {appt_dt}"
    )
    messages = [
        {"role": "system", "content": [{"type": "text",
            "text": prompt_report_writer_system(ehr)}]},
        {"role": "user", "content": [{"type": "text",
            "text": patient_meta_line + "\n\n" + prompt_report_writer_user(
                state.get("interview", {}), ehr)}]},
    ]
    raw    = ask_model(model, messages).strip()
    m      = re.match(r'^\s*```(?:markdown)?\s*(.*?)\s*```\s*$', raw, re.DOTALL | re.IGNORECASE)
    report = m.group(1).strip() if m else raw

    # Post-process: replace any residual "- to be added" placeholders with real values
    _ph_re = re.compile(r'\b[-–]\s*to\s*be\s*added\b', re.IGNORECASE)

    def _fill(match):
        context = report[:match.start()].rsplit('\n', 1)[-1].lower()
        if 'patient' in context or 'name' in context:  return name
        if 'appoint' in context:                        return appt_dt
        if 'date' in context:                           return today
        if 'age' in context:                            return age
        if 'sex' in context or 'gender' in context:     return sex
        return name

    report = _ph_re.sub(_fill, report)
    state["report_summary"] = report

    header = (
        f"### 📋 Clinical Intake Report\n\n"
        f"| | |\n|---|---|\n"
        f"| **Patient Name** | {name} |\n"
        f"| **Age / Sex** | {age} / {sex} |\n"
        f"| **Report Date** | {today} |\n"
        f"| **Appointment Date** | {appt_dt} |\n\n"
        f"---\n\n"
    )
    return state, gr.update(value=header + report, visible=True), gr.update(visible=True)


# ══════════════════════════════════════════════════════════════
#  STEP 2  —  AI Triage
# ══════════════════════════════════════════════════════════════

def step2_begin_triage(state: dict):
    """
    First phase: immediately show loading status while AI runs.
    Returns: (state, triage_md, status_md, dept_dd, confirm_row)
    """
    if not state.get("report_summary"):
        err = "⚠️ Clinical report not found — please complete the Clinical Interview first."
        return (
            state,
            gr.update(value=err),
            gr.update(value=""),
            gr.update(choices=[], value=None),
            gr.update(visible=False),
        )
    return (
        state,
        gr.update(value=""),
        gr.update(value="⏳ **AI triage running …** Analysing patient information, please wait."),
        gr.update(choices=[], value=None),
        gr.update(visible=False),
    )


def step2_run_triage(state: dict):
    """
    Second phase (called via .then()): execute AI triage and show full result
    including a PROVISIONAL routing card so clinician sees everything before confirming.
    Returns: (state, triage_md, status_md, dept_dd, confirm_row)
    """
    model = _get_model()
    name  = state["patient_details"].get("name", "Patient")
    depts = _get_departments()

    triage = run_ai_triage(
        model, name,
        state.get("ehr_summary", ""),
        state.get("report_summary", ""),
        depts,
    )
    state["triage_result"] = triage

    ai_depts = triage.get("departments", [])
    priority = triage.get("triage_priority", "--")
    summary  = triage.get("clinical_summary", "--")
    imaging  = [i.get("name", "") for i in triage.get("recommended_imaging", [])]
    labs_r   = [l.get("name", "") for l in triage.get("recommended_labs", [])]

    # ── Provisional slot lookup ───────────────────────────────
    slot = None
    for dept in ai_depts:
        slots = get_free_slots_from_registry(dept)
        if slots:
            slot = slots[0]
            break
    if not slot:
        slot = get_any_free_slot() or {}

    # Store provisional slot so downstream steps can use it
    state["assigned_slot"]        = slot
    state["selected_departments"] = ai_depts
    cxr_triggered                 = any(d in DEPARTMENT_TO_MODALITIES for d in ai_depts)
    state["cxr_triggered"]        = cxr_triggered

    # CXR mode detection
    cxr_mode = "single"
    if cxr_triggered and state.get("last_visit", 0) > 0:
        if any(d in DEPARTMENT_TO_MODALITIES for d in ai_depts):
            cxr_mode = "longitudinal"
    state["cxr_mode"] = cxr_mode

    # ── Build triage result + provisional routing card ────────
    md  = f"#### 🤖 AI Triage Assessment\n\n"
    md += f"| | |\n|---|---|\n"
    md += f"| **Patient** | {name} |\n"
    md += f"| **Triage Priority** | `{priority}` |\n"
    md += f"| **Recommended Departments** | {', '.join(ai_depts) or '--'} |\n"
    md += f"| **Clinical Summary** | {summary} |\n"
    if imaging:
        md += f"| **Imaging Ordered** | {', '.join(imaging)} |\n"
    if labs_r:
        md += f"| **Labs Ordered** | {', '.join(labs_r)} |\n"

    # Show routing card BEFORE the confirm button
    routing_preview = _build_routing_card(slot, ai_depts)
    md += routing_preview

    if cxr_triggered:
        img_hint = "longitudinal (prior + current X-ray)" if cxr_mode == "longitudinal" else "single X-ray"
        md += (
            f"\n> 🩻 **Imaging required** — {img_hint}. "
            f"Confirm to proceed to AI Diagnosis."
        )
    else:
        md += "\n> 💊 **No imaging required.** Confirm to proceed to Prescription Safety."

    return (
        state,
        gr.update(value=md),
        gr.update(value="✅ **AI triage complete.** Review the assessment and routing above, then confirm below."),
        gr.update(choices=depts, value=ai_depts[0] if ai_depts else None),
        gr.update(visible=True),
    )


def step2_confirm_triage(decision: str, override_dept: str, state: dict):
    """
    Clinician approves or overrides triage.
    Routing card was already shown in triage_result_md.
    Returns: (state, routing_out_msg, tabs_nav)
    """
    triage = state.get("triage_result", {})
    depts  = _get_departments()

    if decision == "Override" and override_dept and override_dept in depts:
        triage["departments"] = [override_dept]
        # Re-lookup slot for overridden dept
        new_slots = get_free_slots_from_registry(override_dept)
        state["assigned_slot"]        = new_slots[0] if new_slots else get_any_free_slot() or {}
        state["selected_departments"] = [override_dept]
        cxr_triggered                 = override_dept in DEPARTMENT_TO_MODALITIES
        state["cxr_triggered"]        = cxr_triggered

        cxr_mode = "single"
        if cxr_triggered and state.get("last_visit", 0) > 0:
            cxr_mode = "longitudinal"
        state["cxr_mode"] = cxr_mode

    triage["human_approved"] = (decision == "Approve")
    state["triage_result"]   = triage
    state["triage_approved"] = triage["human_approved"]

    selected_depts = state["selected_departments"]
    cxr_triggered  = state["cxr_triggered"]
    next_tab       = 3 if cxr_triggered else 4

    dest = "④ AI Diagnosis" if cxr_triggered else "⑤ Prescription Safety"
    routing_msg = (
        f"**{'✅ Approved' if decision == 'Approve' else '🔄 Overridden'}** — "
        f"Routing to **{dest}** …"
    )
    return state, gr.update(value=routing_msg, visible=True), gr.update(selected=next_tab)


# ══════════════════════════════════════════════════════════════
#  STEP 3  —  AI Diagnosis  (CXR + Lab Analysis)
# ══════════════════════════════════════════════════════════════

def step3_prepare_imaging_ui(state: dict):
    """Auto-called on arrival: set mode label + show/hide prior image upload."""
    mode = state.get("cxr_mode", "single")
    if mode == "longitudinal":
        label = (
            "**Longitudinal Mode** — Patient has a prior imaging visit. "
            "Upload **both** the prior and current X-ray for comparative analysis."
        )
        show_prior = True
    else:
        label      = "**Single Mode** — Upload the current chest X-ray below."
        show_prior = False
    return state, gr.update(value=label), gr.update(visible=show_prior)


def step3_analyse(cxr_image_current, cxr_image_prior, lab_text: str, state: dict):
    """Analyse CXR image(s) + lab text and build AI Insight block in state."""
    model = _get_model()
    mode  = state.get("cxr_mode", "single")

    cxr_paths = []
    for img in ([cxr_image_prior, cxr_image_current] if mode == "longitudinal"
                else [cxr_image_current]):
        if img is not None:
            tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
            Image.fromarray(img.astype("uint8")).save(tmp.name)
            cxr_paths.append(tmp.name)
    state["cxr_paths"] = cxr_paths

    raw_out    = run_cxr_analysis(cxr_paths, mode, model) if cxr_paths else "[No image — analysis skipped]"
    cxr_struct = parse_cxr_structured_output(raw_out)
    state["cxr_findings"]   = raw_out
    state["cxr_structured"] = cxr_struct

    lab_parsed = {"labs": []}
    lab_fusion = {"fused_ddx": cxr_struct.get("ddx", []), "fusion_summary": ""}
    if lab_text and lab_text.strip():
        lab_parsed = parse_lab_text(lab_text, model)
        lab_fusion = run_lab_ddx_fusion(
            model, cxr_struct.get("ddx", []), lab_parsed, state.get("ehr_summary", ""))
    state["lab_results_text"]    = lab_text or ""
    state["lab_results_parsed"]  = lab_parsed
    state["lab_fusion_result"]   = lab_fusion

    ddx_list = lab_fusion.get("fused_ddx") or cxr_struct.get("ddx", [])
    rois     = cxr_struct.get("regions_of_interest", [])
    ai_insight = {
        "cxr_mode":                mode,
        "findings":                cxr_struct.get("findings", raw_out),
        "regions_of_interest":     rois,
        "differential_diagnoses":  ddx_list,
        "fusion_summary":          lab_fusion.get("fusion_summary", ""),
        "lab_results":             lab_parsed.get("labs", []),
        "top_diagnosis":           ddx_list[0] if ddx_list else {},
    }
    state["ai_insight"] = ai_insight

    findings = cxr_struct.get("findings", raw_out)
    md  = f"#### 🔬 AI Diagnosis — Imaging & Lab Analysis *(mode: {mode})*\n\n"
    md += f"{findings[:600]}{'...' if len(findings) > 600 else ''}\n"

    if rois:
        md += f"\n**Regions of Interest ({len(rois)}):**\n\n"
        md += "| Finding | Location | Size | Change |\n|---------|----------|------|--------|\n"
        for r in rois:
            md += (f"| {r.get('label','')} | {r.get('location','')} | "
                   f"{r.get('size','')} | {r.get('change','')} |\n")

    if ddx_list:
        md += f"\n**Differential Diagnoses:**\n\n"
        md += "| # | Condition | ICD-10 | Probability | Evidence |\n|---|-----------|--------|-------------|----------|\n"
        for i, d in enumerate(ddx_list[:5], 1):
            prob = d.get("probability", 0)
            ps   = f"{prob:.0%}" if isinstance(prob, (int, float)) else str(prob)
            md  += (f"| {i} | {d.get('condition','')} | {d.get('icd10','')} | "
                    f"{ps} | {d.get('evidence','')[:80]} |\n")

    fsumm = lab_fusion.get("fusion_summary", "")
    if fsumm:
        md += f"\n**Diagnostic Fusion Summary:**\n> {fsumm[:400]}\n"

    labs = lab_parsed.get("labs", [])
    if labs:
        md += f"\n**Lab Results ({len(labs)}):**\n\n"
        md += "| Test | Value | Unit | Ref Range | Status |\n|------|-------|------|-----------|--------|\n"
        for lab in labs[:12]:
            sym = {"high": "⬆️", "low": "⬇️", "critical": "🚨"}.get(
                lab.get("status", "").lower(), "✔️")
            md += (f"| {lab.get('name','')} | {lab.get('value','')} | "
                   f"{lab.get('unit','')} | {lab.get('reference_range','')} | "
                   f"{sym} {lab.get('status','')} |\n")

    md += _build_routing_card(state.get("assigned_slot", {}), state.get("selected_departments", []))

    return state, gr.update(value=md, visible=True), gr.update(visible=True)


# ══════════════════════════════════════════════════════════════
#  STEP 4  —  Prescription Safety
# ══════════════════════════════════════════════════════════════

def step4_load_summary(state: dict):
    """Load full patient summary including AI Insight, for display in Prescription tab."""
    info  = state.get("patient_details", {})
    name  = info.get("name", "--")
    today = datetime.date.today().strftime("%d %B %Y")
    appt  = state.get("appt_date", today)

    md  = f"#### 📄 Patient Clinical Summary\n\n"
    md += f"| | |\n|---|---|\n"
    md += f"| **Name** | {name} |\n"
    md += f"| **ID** | {state.get('patient_id','--')} |\n"
    md += f"| **Age / Sex** | {info.get('age','--')} / {info.get('sex','--')} |\n"
    md += f"| **Report Date** | {today} |\n"
    md += f"| **Appointment Date** | {appt} |\n"
    md += f"| **Prior Visits** | {state.get('last_visit', 0)} |\n"

    pgx = info.get("gene_profile", {})
    if pgx:
        md += f"| **Gene Profile** | {pgx} |\n"

    ehr = state.get("ehr_summary", "")
    if ehr and ehr != "There is No Past Records":
        md += f"\n**EHR Summary:**\n> {ehr[:500]}{'...' if len(ehr)>500 else ''}\n"

    report = state.get("report_summary", "")
    if report:
        md += f"\n**Clinical Intake Report:**\n> {report[:500]}{'...' if len(report)>500 else ''}\n"

    tr = state.get("triage_result", {})
    if tr:
        md += (f"\n**Triage:** `{tr.get('triage_priority','--')}` | "
               f"Departments: {tr.get('departments',[])} | "
               f"Approved: {'Yes' if tr.get('human_approved') else 'No'}\n")

    ai_insight = state.get("ai_insight", {})
    if ai_insight:
        md += f"\n---\n#### 🤖 AI Insight\n"
        top = ai_insight.get("top_diagnosis", {})
        if top:
            prob = top.get("probability", 0)
            ps   = f" ({prob:.0%})" if isinstance(prob, (int, float)) else ""
            md  += f"**Top Diagnosis:** {top.get('condition','--')} — `{top.get('icd10','')}`{ps}\n\n"

        ddx = ai_insight.get("differential_diagnoses", [])
        if ddx:
            md += "**Differential Diagnoses:**\n\n"
            md += "| Condition | ICD-10 | Probability |\n|-----------|--------|-------------|\n"
            for d in ddx[:3]:
                prob = d.get("probability", 0)
                ps   = f"{prob:.0%}" if isinstance(prob, (int, float)) else str(prob)
                md  += f"| {d.get('condition','')} | {d.get('icd10','')} | {ps} |\n"

        rois = ai_insight.get("regions_of_interest", [])
        if rois:
            labels = ", ".join(r.get("label", "") for r in rois[:3])
            md += f"\n**Regions of Interest:** {labels}\n"

        fsumm = ai_insight.get("fusion_summary", "")
        if fsumm:
            md += f"\n**Fusion Summary:** {fsumm[:300]}\n"

    cond = info.get("conditions", [])
    if cond:
        md += f"\n**Known Conditions:** {', '.join(cond)}\n"
    meds = info.get("current_medications", [])
    if meds:
        names = [m.get("name", str(m)) if isinstance(m, dict) else str(m) for m in meds]
        md += f"**Current Medications:** {', '.join(names)}\n"
    allergies = info.get("allergies", [])
    if allergies:
        alg = [a.get("substance", str(a)) if isinstance(a, dict) else str(a) for a in allergies]
        md += f"**Allergies:** {', '.join(alg)}\n"

    slot = state.get("assigned_slot", {})
    if slot:
        md += _build_routing_card(slot, state.get("selected_departments", []))

    return gr.update(value=md)


def step4_check_drug(drug: str, dosage: str, indication: str, state: dict):
    drug = (drug or "").strip()
    if not drug:
        return (
            state,
            gr.update(value="⚠️ Please enter a proposed drug name.", visible=True),
            gr.update(visible=False),
        )

    info = state.get("patient_details", {})
    ctx  = {
        "allergies":           info.get("allergies", []),
        "current_medications": info.get("current_medications", []),
        "conditions":          info.get("conditions", []),
        "renal_function":      info.get("renal_function", {"egfr": 999, "status": "NORMAL"}),
        "gene_profile":        info.get("gene_profile", {}),
    }
    result = run_drug_safety_check(drug, dosage, indication, ctx)

    # ── AI pharmacological review — catches dosage/indication issues ──────
    # The KB-only check only fires when patient has documented history.
    # The AI independently evaluates dose appropriateness & standard safety.
    try:
        _ai_prompt = (
            "You are a senior clinical pharmacologist. Strictly evaluate:\n\n"
            f"Drug: {drug}\n"
            f"Dosage: {dosage or 'not specified'}\n"
            f"Indication: {indication or 'not specified'}\n"
            f"Patient: Age {info.get('age','--')}, Sex {info.get('sex','--')}\n\n"
            "Evaluate:\n"
            "1. Is the dosage too HIGH, too LOW, or appropriate for this drug?\n"
            "2. Are there standard contraindications for this drug regardless of patient history?\n"
            "3. Does the indication match the drug\'s approved clinical uses?\n"
            "4. Any serious adverse effects at this dose that require a warning or block?\n\n"
            "Respond ONLY with compact JSON — no prose, no markdown fences:\n"
            '{"dosage_ok": true/false, "concerns": [{"type": "Dosage|Contraindication|Indication Mismatch|Safety", '
            '"action": "BLOCK|WARN", "severity": "CRITICAL|HIGH|MODERATE|LOW", '
            '"detail": "...", "recommendation": "..."}], "summary": "..."}\n'
            'If nothing is concerning return: {"dosage_ok": true, "concerns": [], "summary": "Dosage and indication appear appropriate."}'
        )
        _ai_msgs = [
            {"role": "system", "content": [{"type": "text",
                "text": "You are a clinical pharmacologist AI. Output ONLY valid compact JSON."}]},
            {"role": "user", "content": [{"type": "text", "text": _ai_prompt}]},
        ]
        _ai_raw = ask_model(_get_model(), _ai_msgs).strip()
        _m = re.search(r"\{.*\}", _ai_raw, re.DOTALL)
        _ai_parsed = json.loads(_m.group()) if _m else {}
        for _c in _ai_parsed.get("concerns", []):
            _entry = {
                "type":           _c.get("type", "AI Clinical Review"),
                "action":         _c.get("action", "WARN"),
                "severity":       _c.get("severity", "MODERATE"),
                "detail":         _c.get("detail", ""),
                "recommendation": _c.get("recommendation", ""),
            }
            if _c.get("action") == "BLOCK":
                result["blocks"].append(_entry)
                result["status"] = "BLOCK"
            else:
                result.setdefault("warnings", []).append(_entry)
        # Recalculate alert_level after merging AI findings
        if result.get("blocks"):
            result["alert_level"] = max(
                result["blocks"], key=lambda v: _severity_rank(v.get("severity", "LOW"))
            ).get("severity", "HIGH")
        elif result.get("warnings"):
            result["alert_level"] = max(
                result["warnings"], key=lambda v: _severity_rank(v.get("severity", "LOW"))
            ).get("severity", "LOW")
        result["ai_drug_summary"] = _ai_parsed.get("summary", "")
    except Exception:
        pass  # AI review non-fatal — KB result still used

    state.update({
        "proposed_drug":      drug,
        "proposed_dosage":    dosage,
        "drug_indication":    indication,
        "drug_safety_result": result,
    })

    status_val = result.get("status") or "PASS"
    alert_raw  = result.get("alert_level")
    alert      = str(alert_raw).upper() if alert_raw not in (None, "", "None") else "NONE"
    icon       = "🚫 BLOCKED" if status_val == "BLOCK" else "✅ APPROVED"

    md  = f"#### 💊 Drug Safety Assessment\n\n"
    md += f"| | |\n|---|---|\n"
    md += f"| **Drug** | {drug} {dosage} |\n"
    md += f"| **Indication** | {indication} |\n"
    md += f"| **Status** | **{icon}** |\n"
    md += f"| **Alert Level** | `{alert}` |\n"

    for b in result.get("blocks", []):
        md += f"\n---\n🚫 **BLOCK — {b.get('severity','')}**\n{b.get('detail','')}\n"
        if b.get("recommendation"):
            md += f"> 💡 *{b['recommendation']}*\n"

    for w in result.get("warnings", []):
        md += f"\n---\n⚠️ **WARNING — {w.get('severity','')}**\n{w.get('detail','')}\n"
        if w.get("recommendation"):
            md += f"> 💡 *{w['recommendation']}*\n"

    for p in result.get("pharmacogenomics", []):
        md += (f"\n---\n🧬 **PGx — {p.get('gene','')} / {p.get('patient_phenotype','')}**\n"
               f"{p.get('detail','')[:200]}\n")
        if p.get("recommendation"):
            md += f"> 💡 *{p['recommendation']}*\n"

    if not result.get("blocks") and not result.get("warnings") and not result.get("pharmacogenomics"):
        md += "\n✅ No drug interactions, contraindications, or pharmacogenomics concerns found.\n"

    ai_summ = result.get("ai_drug_summary", "")
    if ai_summ:
        md += f"\n---\n🤖 **AI Pharmacological Review:**\n{ai_summ}\n"

    return state, gr.update(value=md, visible=True), gr.update(visible=True)


# ══════════════════════════════════════════════════════════════
#  STEP 5  —  Complete Records
# ══════════════════════════════════════════════════════════════

def step5_finalise(follow_up: bool, follow_up_date,
                   save_json_flag: bool, save_fhir_flag: bool, state: dict):
    """
    Save session records, generate download txt, and display full patient record.
    All IO is wrapped in try-except so a save error never prevents the summary from rendering.
    Returns:
      (state, summary_md, session_json_str, fhir_json_str,
       fhir_acc, json_acc, download_file)
    """
    pid     = state.get("patient_id", "unknown")
    details = state.get("patient_details", {})

    # ── Normalise follow-up date ──────────────────────────────
    fu_date_raw = follow_up_date
    # datetime.datetime object (from gr.DateTime calendar)
    if hasattr(fu_date_raw, "strftime"):
        fu_date_raw = fu_date_raw.strftime("%d %B %Y")
    # datetime.date object
    elif hasattr(fu_date_raw, "isoformat"):
        fu_date_raw = str(fu_date_raw)
    else:
        fu_date_raw = (str(fu_date_raw) if fu_date_raw else "").strip()

    state["follow_up_required"] = follow_up
    state["follow_up_date"]     = fu_date_raw if follow_up else ""

    cxr_struct  = state.get("cxr_structured", {})
    lab_parsed  = state.get("lab_results_parsed", {})
    lab_fusion  = state.get("lab_fusion_result", {})
    drug_safety = state.get("drug_safety_result", {})
    ai_insight  = state.get("ai_insight", {})
    slot        = state.get("assigned_slot", {})

    doctor = slot.get("doctor_name", "--")
    dept   = slot.get("department", "--")
    queue  = slot.get("slot_id", "--")
    hosp   = slot.get("hospital_name", "--")
    appt   = state.get("appt_date", "--")

    # ── Build record dict ─────────────────────────────────────
    save_errors = []
    try:
        rec = build_patient_json_record(
            patient_id      = pid,
            patient_details = details,
            ehr_summary     = state.get("ehr_summary", ""),
            interview       = state.get("interview", {}),
            report_summary  = state.get("report_summary", ""),
            triage          = state.get("triage_result", {}),
            booking         = {
                "department": dept, "doctor": doctor,
                "queue": queue,     "hospital": hosp,
                "appointment_date": appt,
            },
            imaging = {
                "mode":       state.get("cxr_mode", ""),
                "paths":      state.get("cxr_paths", []),
                "findings":   state.get("cxr_findings", ""),
                "structured": cxr_struct,
                "lab_parsed": lab_parsed,
                "lab_fusion": lab_fusion,
                "ai_insight": ai_insight,
            },
            drug_safety = drug_safety,
            visit       = state.get("last_visit", 0) + 1,
        )
    except Exception as _e:
        save_errors.append(f"Record build: {_e}")
        rec = {
            "meta": {"session_id": str(uuid.uuid4()), "patient_id": pid,
                     "visit": state.get("last_visit", 0) + 1,
                     "date_time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")},
            "demographics": details,
        }

    rec.setdefault("meta", {})
    rec["meta"]["follow_up_required"] = follow_up
    rec["meta"]["follow_up_date"]     = state["follow_up_date"]

    sid = rec["meta"].get("session_id", str(uuid.uuid4()))[:8]
    state["session_id"] = sid

    # ── Save JSON ─────────────────────────────────────────────
    session_json_str = ""
    out_path         = ""
    if save_json_flag:
        try:
            make_directory(MOD1_CONVERSATION_DIR)
            out_path = MOD1_CONVERSATION_DIR / f"session_{pid}_{sid}.json"
            save_json(rec, str(out_path))
            if lab_parsed.get("labs"):
                make_directory(LAB_SESSIONS_DIR)
                save_json(lab_parsed, str(LAB_SESSIONS_DIR / f"{pid}_{sid}_labs.json"))
            session_json_str = json.dumps(rec, indent=2)
        except Exception as _e:
            save_errors.append(f"JSON save: {_e}")

    # ── Save FHIR ─────────────────────────────────────────────
    fhir_json_str = ""
    fhir_path     = ""
    if save_fhir_flag:
        try:
            fhir_res = [build_fhir_patient_resource(details)]
            for ddx_e in cxr_struct.get("ddx", []):
                fhir_res.append(build_fhir_condition_resource(ddx_e, pid))
            for lab in lab_parsed.get("labs", []):
                fhir_res.append(build_fhir_observation_resource(lab, pid))
            if drug_safety.get("proposed_drug"):
                fhir_res.append(build_fhir_medication_request_resource(
                    drug_safety["proposed_drug"], drug_safety.get("dosage", ""), drug_safety, pid))
            fhir_path = build_fhir_bundle(pid, sid, *fhir_res)
            state["fhir_bundle_path"] = fhir_path
            fhir_obj = read_json(fhir_path) if fhir_path else {}
            fhir_json_str = json.dumps(fhir_obj, indent=2)[:6000]
        except Exception as _e:
            save_errors.append(f"FHIR save: {_e}")

    name  = details.get("name", "--")
    today = datetime.date.today().strftime("%d %B %Y")
    ddx_out = lab_fusion.get("fused_ddx") or cxr_struct.get("ddx", [])

    # ── Build summary display markdown ───────────────────────
    md  = f"## ✅ Clinical Record {'Saved' if not save_errors else 'Processed'}\n\n"
    md += f"**Patient:** {name}  |  **Session:** `{sid}`  |  **Date:** {today}\n\n"
    md += (
        f"### 🏥 Clinic Assignment\n\n"
        f"| | |\n|---|---|\n"
        f"| **Department** | {dept} |\n"
        f"| **Doctor** | {doctor} |\n"
        f"| **Hospital** | {hosp} |\n"
        f"| **Appointment Date** | {appt} |\n"
        f"| **Queue / Slot** | `{queue}` |\n\n"
    )

    if ddx_out:
        md += "### 🔬 Top Diagnoses\n\n"
        md += "| # | Condition | ICD-10 | Probability |\n|---|-----------|--------|-------------|\n"
        for i, d in enumerate(ddx_out[:3], 1):
            prob = d.get("probability", 0)
            ps   = f"{prob:.0%}" if isinstance(prob, (int, float)) else str(prob)
            md  += f"| {i} | {d.get('condition','')} | {d.get('icd10','')} | {ps} |\n"
        md += "\n"

    dr = state.get("drug_safety_result", {})
    if dr.get("proposed_drug"):
        alert_raw = dr.get("alert_level")
        alert_str = str(alert_raw).upper() if alert_raw not in (None, "", "None") else "NONE"
        icon = "🚫" if dr.get("status") == "BLOCK" else "✅"
        md += (
            f"### 💊 Prescription\n"
            f"{icon} **{dr['proposed_drug']}** {dr.get('dosage','')} — "
            f"{dr.get('status','--')} (Alert: `{alert_str}`)\n\n"
        )

    md += "### 💾 Saved Files\n"
    if save_json_flag and out_path:
        md += f"- Session JSON: `{out_path}`\n"
    if save_fhir_flag and fhir_path:
        md += f"- FHIR R4 Bundle: `{fhir_path}`\n"
    if not save_json_flag and not save_fhir_flag:
        md += "_No files saved (both options unchecked)._\n"
    if save_errors:
        md += "\n⚠️ **Save warnings:**\n"
        for err in save_errors:
            md += f"- {err}\n"

    # ── Follow-up (appended at end of record) ────────────────
    if follow_up:
        fu_display = state["follow_up_date"] or "Date not specified"
        md += f"\n### 📆 Follow-Up Required\n**Scheduled:** {fu_display}\n"
        rec.setdefault("follow_up_note", f"Follow-up required: {fu_display}")
        if save_json_flag and out_path:
            try:
                save_json(rec, str(out_path))
            except Exception:
                pass
    else:
        md += "\n### 📆 Follow-Up\nNot required for this visit.\n"

    # ── Append full patient record to summary ─────────────────
    md += "\n" + _build_overall_record_md(state)

    # ── Save .txt for local download ──────────────────────────
    txt_path = ""
    txt_file = None
    try:
        make_directory(MOD1_CONVERSATION_DIR)
        txt_file = MOD1_CONVERSATION_DIR / f"session_{pid}_{sid}_summary.txt"
        save_txt(md, str(txt_file))
        txt_path = str(txt_file)
        state["saved_txt_path"] = txt_path
    except Exception as _e:
        save_errors.append(f"TXT export: {_e}")

    # ── Update MOD1_CSV_PATH so return visits show last-visit info ──
    if save_json_flag and (out_path or txt_path):
        try:
            csv_row = {
                "patient_id":        pid,
                "date_time":         datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                "visit":             rec["meta"].get("visit", state.get("last_visit", 0) + 1),
                "status":            "complete",
                "priority":          state.get("triage_result", {}).get("triage_priority", "STABLE").lower(),
                "target_department": ", ".join(state.get("selected_departments", [])),
                "conversation_path": "",
                "report_path":       txt_path or str(out_path),
            }
            if MOD1_CSV_PATH.exists():
                csv_df = pd.read_csv(MOD1_CSV_PATH, dtype=str)
                # Ensure all required columns exist
                for col in csv_row:
                    if col not in csv_df.columns:
                        csv_df[col] = ""
            else:
                csv_df = pd.DataFrame(columns=list(csv_row.keys()))
                make_directory(MOD1_CSV_PATH.parent)
            csv_df = pd.concat([csv_df, pd.DataFrame([csv_row])], ignore_index=True)
            csv_df.to_csv(str(MOD1_CSV_PATH), index=False)
        except Exception as _e:
            save_errors.append(f"Visit log: {_e}")

    return (
        state,
        gr.update(value=md, visible=True),
        session_json_str,
        fhir_json_str,
        gr.update(visible=save_fhir_flag and bool(fhir_path)),
        gr.update(visible=save_json_flag and bool(out_path)),
        gr.update(value=txt_path if txt_path else None, visible=bool(txt_path)),
    )


def step5_toggle_followup(checked: bool):
    """Show or hide the follow-up date input."""
    return gr.update(visible=checked)


# ══════════════════════════════════════════════════════════════
#  UI  —  launch_gradio_ui()
# ══════════════════════════════════════════════════════════════

def launch_gradio_ui(preload: bool = True):
    """
    Build and launch the Continuum sequential clinical pipeline UI.
    Set preload=True (default) to warm up models before the first patient.
    """
    if preload:
        preload_pipeline()

    _css = """
@import url('https://fonts.googleapis.com/css2?family=IBM+Plex+Sans:ital,wght@0,300;0,400;0,500;0,600;0,700;1,400&display=swap');

* {
    font-family: 'IBM Plex Sans', system-ui, sans-serif !important;
}
.gradio-container, .gradio-container * {
    font-family: 'IBM Plex Sans', system-ui, sans-serif !important;
}
code, pre, .code, .codehilite, textarea.code {
    font-family: 'IBM Plex Mono', 'Fira Code', monospace !important;
}
"""
    with gr.Blocks(title="Continuum", theme=gr.themes.Soft(), css=_css) as demo:

        pipeline_state = gr.State(_fresh_state())

        gr.Markdown(
            "# 🩺 Continuum — Clinical AI Platform\n"
            "Proceed through each tab in order. "
            "**Next →** buttons appear only after the current step is complete."
        )

        with gr.Tabs(selected=0) as tabs:

            # ═══════════════════════════════════════════════
            # TAB 0  ·  Registration
            # ═══════════════════════════════════════════════
            with gr.Tab("① Registration", id=0):
                gr.Markdown(
                    "### Patient Registration\n"
                    "Enter the patient mobile number. "
                    "Existing patients load instantly; new patients fill in the form below."
                )
                with gr.Row():
                    phone_in = gr.Textbox(
                        label="Mobile Number", placeholder="+60123456789", scale=4)
                    find_btn = gr.Button("🔍 Find Patient", variant="primary", scale=1)

                patient_card = gr.Markdown(value="")
                status_msg   = gr.Markdown(value="")

                with gr.Row(visible=False) as new_patient_row:
                    new_name_in  = gr.Textbox(label="Full Name", placeholder="e.g. Tony Stark")
                    new_age_in   = gr.Textbox(label="Age",       placeholder="e.g. 45")
                    new_sex_in   = gr.Dropdown(
                        choices=["Male", "Female", "Other"], label="Sex")
                    register_btn = gr.Button("📋 Register Patient", variant="secondary")

                to_interview_btn = gr.Button(
                    "Begin Clinical Interview →", variant="primary", visible=False)

            # ═══════════════════════════════════════════════
            # TAB 1  ·  Clinical Interview
            # ═══════════════════════════════════════════════
            with gr.Tab("② Clinical Interview", id=1):
                gr.Markdown(
                    "### Clinical Interview\n"
                    "The AI assistant asks one question at a time. "
                    "Type your answer and click **Send** or press Enter."
                )
                interview_status = gr.Markdown(value="")
                chatbot = gr.Chatbot(
                    label="Clinical Assistant", height=400,
                    type="messages", bubble_full_width=False)
                with gr.Row():
                    answer_box = gr.Textbox(
                        label="Your answer",
                        placeholder="Interview will start automatically when you arrive here …",
                        interactive=False, scale=5)
                    send_btn = gr.Button("Send ➤", variant="primary", scale=1)

                generate_report_btn = gr.Button(
                    "📝 Generate Clinical Report", variant="secondary", visible=False)
                report_status_md = gr.Markdown(value="")
                report_out       = gr.Markdown(value="", visible=False)
                to_triage_btn    = gr.Button(
                    "Proceed to AI Triage →", variant="primary", visible=False)

            # ═══════════════════════════════════════════════
            # TAB 2  ·  AI Triage
            # ═══════════════════════════════════════════════
            with gr.Tab("③ AI Triage", id=2):
                gr.Markdown(
                    "### AI Triage Assessment\n"
                    "AI assesses the patient and recommends departments with a **routing card** "
                    "shown below. Review the routing and confirm or override."
                )
                triage_status_md = gr.Markdown(value="")
                triage_result_md = gr.Markdown(value="")

                with gr.Row(visible=False) as confirm_row:
                    with gr.Column(scale=1):
                        approve_radio = gr.Radio(
                            ["Approve", "Override"], value="Approve", label="Decision")
                    with gr.Column(scale=2):
                        dept_override_dd = gr.Dropdown(
                            choices=[], label="Override to Department",
                            info="Select only when choosing Override", interactive=True)
                    with gr.Column(scale=1):
                        confirm_triage_btn = gr.Button(
                            "✅ Confirm & Route →", variant="primary")

                routing_out = gr.Markdown(value="", visible=False)

            # ═══════════════════════════════════════════════
            # TAB 3  ·  AI Diagnosis
            # ═══════════════════════════════════════════════
            with gr.Tab("④ AI Diagnosis", id=3):
                gr.Markdown(
                    "### AI Diagnosis — Imaging & Lab Analysis\n"
                    "Upload the chest X-ray image(s) and optional lab results."
                )
                imaging_mode_md = gr.Markdown(value="")

                with gr.Row(visible=False) as prior_image_row:
                    cxr_upload_prior = gr.Image(
                        label="📁 Prior Visit X-Ray (longitudinal)", type="numpy")

                with gr.Row():
                    cxr_upload_current = gr.Image(
                        label="📷 Current X-Ray", type="numpy", scale=1)
                    lab_text_in = gr.Textbox(
                        label="Lab Results (optional — paste raw text)",
                        lines=10, placeholder="Paste lab report text here …", scale=1)

                imaging_status_md = gr.Markdown(value="")
                analyse_btn       = gr.Button("▶ Run AI Diagnosis", variant="primary")
                imaging_result    = gr.Markdown(value="", visible=False)
                to_rx_btn         = gr.Button(
                    "Proceed to Prescription Safety →", variant="primary", visible=False)

            # ═══════════════════════════════════════════════
            # TAB 4  ·  Prescription Safety
            # ═══════════════════════════════════════════════
            with gr.Tab("⑤ Prescription Safety", id=4):
                gr.Markdown(
                    "### Prescription Safety Check\n"
                    "Review the patient summary (with AI Insight) then enter the proposed prescription."
                )
                patient_summary_out = gr.Markdown(
                    value="*Patient summary with AI Insight will load automatically.*")

                gr.Markdown("---\n#### Prescription Input")
                with gr.Row():
                    drug_in   = gr.Textbox(label="Proposed Drug",         placeholder="e.g. Metformin")
                    dosage_in = gr.Textbox(label="Dosage",                 placeholder="e.g. 500mg BD")
                    ind_in    = gr.Textbox(label="Indication / Diagnosis", placeholder="e.g. Type 2 Diabetes")

                drug_status_md = gr.Markdown(value="")
                check_drug_btn = gr.Button("▶ Check Prescription Safety", variant="primary")
                drug_result    = gr.Markdown(value="", visible=False)
                to_records_btn = gr.Button(
                    "Proceed to Complete Records →", variant="primary", visible=False)

            # ═══════════════════════════════════════════════
            # TAB 5  ·  Complete Records
            # ═══════════════════════════════════════════════
            with gr.Tab("⑥ Complete Records", id=5):
                gr.Markdown(
                    "### Complete Clinical Record\n"
                    "Save the session. Clinic assignment was determined during triage — "
                    "no separate booking required."
                )
                with gr.Row():
                    follow_up_chk = gr.Checkbox(
                        label="📆 Follow-up required", value=False, scale=1)
                    # Calendar date-picker (Gradio 4.x gr.DateTime, date-only)
                    follow_up_date_in = gr.DateTime(
                        label="📅 Follow-up Date",
                        include_time=False,
                        visible=False,
                        scale=2,
                    )

                gr.Markdown("#### Save Options")
                with gr.Row():
                    save_json_chk = gr.Checkbox(
                        label="💾 Save Session JSON",   value=True, scale=1)
                    save_fhir_chk = gr.Checkbox(
                        label="🏥 Save FHIR R4 Bundle", value=True, scale=1)

                save_status_md = gr.Markdown(value="")
                finalise_btn   = gr.Button("▶ Save All Records", variant="primary")
                final_summary  = gr.Markdown(value="", visible=False)

                # Download button — visible only after a successful save
                download_txt = gr.File(
                    label="⬇ Download Summary (.txt)",
                    visible=False,
                    file_count="single",
                )

                with gr.Accordion("📄 Session JSON", open=False, visible=False) as json_acc:
                    session_json_out = gr.Code(language="json", label="Session Record")
                with gr.Accordion("🏥 FHIR R4 Bundle", open=False, visible=False) as fhir_acc:
                    fhir_json_out = gr.Code(language="json", label="FHIR Bundle")

        # ═══════════════════════════════════════════════════════
        #  WIRING
        # ═══════════════════════════════════════════════════════

        # ── Tab 0  —  Registration ──────────────────────────────────
        find_btn.click(
            fn=lambda: gr.update(value="⏳ Looking up patient …"),
            outputs=[status_msg],
        ).then(
            fn=step0_find_patient,
            inputs=[phone_in, pipeline_state],
            outputs=[pipeline_state, patient_card,
                     to_interview_btn, new_patient_row, status_msg],
        )
        phone_in.submit(
            fn=lambda: gr.update(value="⏳ Looking up patient …"),
            outputs=[status_msg],
        ).then(
            fn=step0_find_patient,
            inputs=[phone_in, pipeline_state],
            outputs=[pipeline_state, patient_card,
                     to_interview_btn, new_patient_row, status_msg],
        )
        register_btn.click(
            fn=step0_register_new_patient,
            inputs=[phone_in, new_name_in, new_age_in, new_sex_in, pipeline_state],
            outputs=[pipeline_state, patient_card, to_interview_btn, new_patient_row],
        )

        # ── Tab 0→1  (navigate then auto-start interview) ──────────
        to_interview_btn.click(
            fn=lambda: (gr.update(selected=1),
                        gr.update(value="⏳ Loading interview …")),
            outputs=[tabs, interview_status],
        ).then(
            fn=step1_begin_interview,
            inputs=[pipeline_state],
            outputs=[pipeline_state, chatbot, answer_box,
                     generate_report_btn, interview_status],
        )

        # ── Tab 1  —  Interview Q&A ─────────────────────────────────
        send_btn.click(
            fn=step1_send_answer,
            inputs=[answer_box, chatbot, pipeline_state],
            outputs=[pipeline_state, chatbot, answer_box, generate_report_btn],
        )
        answer_box.submit(
            fn=step1_send_answer,
            inputs=[answer_box, chatbot, pipeline_state],
            outputs=[pipeline_state, chatbot, answer_box, generate_report_btn],
        )

        # Report generation — two-phase loading
        generate_report_btn.click(
            fn=lambda: (
                gr.update(value="⏳ Generating clinical report …", visible=True),
                gr.update(visible=False),
                gr.update(visible=False),
            ),
            outputs=[report_status_md, report_out, to_triage_btn],
        ).then(
            fn=step1_generate_report,
            inputs=[pipeline_state],
            outputs=[pipeline_state, report_out, to_triage_btn],
        ).then(
            fn=lambda: gr.update(value=""),
            outputs=[report_status_md],
        )

        # ── Tab 1→2  (navigate then two-phase AI triage) ───────────
        to_triage_btn.click(
            fn=lambda: gr.update(selected=2), outputs=tabs,
        ).then(
            fn=step2_begin_triage,
            inputs=[pipeline_state],
            outputs=[pipeline_state, triage_result_md, triage_status_md,
                     dept_override_dd, confirm_row],
        ).then(
            fn=step2_run_triage,
            inputs=[pipeline_state],
            outputs=[pipeline_state, triage_result_md, triage_status_md,
                     dept_override_dd, confirm_row],
        )

        # ── Tab 2  —  Triage confirm + route ───────────────────────
        confirm_triage_btn.click(
            fn=step2_confirm_triage,
            inputs=[approve_radio, dept_override_dd, pipeline_state],
            outputs=[pipeline_state, routing_out, tabs],
        ).then(
            fn=step4_load_summary,
            inputs=[pipeline_state],
            outputs=[patient_summary_out],
        )
        # Also prepare imaging UI (second handler on same button)
        confirm_triage_btn.click(
            fn=step3_prepare_imaging_ui,
            inputs=[pipeline_state],
            outputs=[pipeline_state, imaging_mode_md, prior_image_row],
        )

        # ── Tab 3  —  AI Diagnosis — two-phase loading ─────────────
        analyse_btn.click(
            fn=lambda: (
                gr.update(value="⏳ Running AI diagnosis, please wait …"),
                gr.update(visible=False),
                gr.update(visible=False),
            ),
            outputs=[imaging_status_md, imaging_result, to_rx_btn],
        ).then(
            fn=step3_analyse,
            inputs=[cxr_upload_current, cxr_upload_prior, lab_text_in, pipeline_state],
            outputs=[pipeline_state, imaging_result, to_rx_btn],
        ).then(
            fn=lambda: gr.update(value="✅ AI diagnosis complete."),
            outputs=[imaging_status_md],
        )

        to_rx_btn.click(
            fn=lambda: gr.update(selected=4), outputs=tabs,
        ).then(
            fn=step4_load_summary,
            inputs=[pipeline_state],
            outputs=[patient_summary_out],
        )

        # ── Tab 4  —  Drug Safety — two-phase loading ──────────────
        check_drug_btn.click(
            fn=lambda: (
                gr.update(value="⏳ Checking prescription safety …"),
                gr.update(visible=False),
                gr.update(visible=False),
            ),
            outputs=[drug_status_md, drug_result, to_records_btn],
        ).then(
            fn=step4_check_drug,
            inputs=[drug_in, dosage_in, ind_in, pipeline_state],
            outputs=[pipeline_state, drug_result, to_records_btn],
        ).then(
            fn=lambda: gr.update(value=""),
            outputs=[drug_status_md],
        )

        to_records_btn.click(
            fn=lambda: gr.update(selected=5), outputs=tabs,
        )

        # ── Tab 5  —  Follow-up toggle ──────────────────────────────
        follow_up_chk.change(
            fn=step5_toggle_followup,
            inputs=[follow_up_chk],
            outputs=[follow_up_date_in],
        )

        # ── Tab 5  —  Save records — two-phase loading ─────────────
        finalise_btn.click(
            fn=lambda: (
                gr.update(value="⏳ Saving records …"),
                gr.update(visible=False),
                gr.update(visible=False),
            ),
            outputs=[save_status_md, final_summary, download_txt],
        ).then(
            fn=step5_finalise,
            inputs=[follow_up_chk, follow_up_date_in,
                    save_json_chk, save_fhir_chk, pipeline_state],
            outputs=[pipeline_state, final_summary,
                     session_json_out, fhir_json_out,
                     fhir_acc, json_acc, download_txt],
        ).then(
            fn=lambda: gr.update(value="✅ Records saved successfully."),
            outputs=[save_status_md],
        )

    demo.queue().launch(server_port=7860, share=False, inbrowser=True)
    return demo


In [11]:
launch_gradio_ui()

Loading MedGemma 4b on cuda …


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


✓ MedGemma 4b loaded


/tmp/ipython-input-384100550.py:1251: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Continuum", theme=gr.themes.Soft(), css=_css) as demo:
/tmp/ipython-input-384100550.py:1251: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Continuum", theme=gr.themes.Soft(), css=_css) as demo:
/tmp/ipython-input-384100550.py:1300: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipython-input-384100550.py:1300: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Cha

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Gradio Blocks instance: 31 backend functions
--------------------------------------------
fn_index=0
 inputs:
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x7fd45e89bcb0>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7fd45f983b60>
 |-<gradio.components.state.State object at 0x7fd467c08680>
 outputs:
 |-<gradio.components.state.State object at 0x7fd467c08680>
 |-<gradio.components.markdown.Markdown object at 0x7fd45de2cec0>
 |-<gradio.components.button.Button object at 0x7fd45f9812e0>
 |-<gradio.layouts.row.Row object at 0x7fd4475df590>
 |-<gradio.components.markdown.Markdown object at 0x7fd45e89bcb0>
fn_index=2
 inputs:
 outputs:
 |-<gradio.components.markdown.Markdown object at 0x7fd45e89bcb0>
fn_index=3
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7fd45f983b60>
 |-<gradio.components.state.State object at 0x7fd467c08680>
 outputs:
 |-<gradio.components.state.State object at 0x7fd467c08680>
 |-<gradio.components.markdown.Markdown obj